# 02 — Data Preparation and Preprocessing

**Mục tiêu:** áp dụng quy tắc xử lý đã xác định, gắn quality flags, chuẩn hóa từng bảng, tổng hợp payment/product/order và tạo dataset đầu ra cho EDA.

> Cell khởi tạo dưới đây giúp notebook chạy độc lập: import thư viện, mount Google Drive, khai báo đường dẫn, khởi tạo Spark và nạp ba bảng lõi `reviews`, `orders`, `customers`.

In [ ]:
%pip -q install pyspark

from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "BigData" / "DA-BigData" / "code" / "data"
PROCESSED_DIR = DATA_DIR / "processed"

data_dir = DATA_DIR
processed_dir = PROCESSED_DIR
customer_path = DATA_DIR / "olist_customers_dataset.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_raw_files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_products_dataset.csv"
]

missing_raw_files = [
    name for name in required_raw_files
    if not (DATA_DIR / name).exists()
]

if missing_raw_files:
    raise FileNotFoundError(
        "Thiếu file dữ liệu gốc trong DATA_DIR: " + ", ".join(missing_raw_files)
    )

spark = (
    SparkSession.builder
    .appName("OlistDataPreparationPreprocessing")
    .getOrCreate()
)

reviews = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(str(DATA_DIR / "olist_order_reviews_dataset.csv"))
)

orders = (
    spark.read
    .option("header", True)
    .csv(str(DATA_DIR / "olist_orders_dataset.csv"))
)

customers = (
    spark.read
    .option("header", True)
    .csv(str(customer_path))
)

reviews.createOrReplaceTempView("reviews")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

# Nếu các bảng prepared đã được tạo ở một lần chạy trước, nạp sẵn để
# các cell validation cuối notebook có thể dùng lại mà không phụ thuộc RAM cũ.
order_payments_prepared_path = PROCESSED_DIR / "olist_order_payments_prepared.csv"
products_prepared_path = PROCESSED_DIR / "olist_products_prepared.csv"

if order_payments_prepared_path.exists():
    order_payments_prepared = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(order_payments_prepared_path))
    )

if products_prepared_path.exists():
    products_prepared = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(products_prepared_path))
    )

print("Đã kết nối Google Drive:", DRIVE_ROOT.exists())
print("DATA_DIR      :", DATA_DIR)
print("PROCESSED_DIR :", PROCESSED_DIR)
print("Spark version :", spark.version)
print("Core tables   : reviews, orders, customers")

Mounted at /content/drive
Đã kết nối Google Drive: True
DATA_DIR      : /content/drive/MyDrive/BigData/DA-BigData/code/data
PROCESSED_DIR : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed
Spark version : 4.0.4
Core tables   : reviews, orders, customers


##Tạo bảng review phục vụ cho huấn luyện:
- Liên kết hợp lệ về thời gian: đơn được đặt trước ngày tạo review. Điều này chưa khẳng định review chắc chắn thuộc đơn đó.
- Liên kết bị loại: đơn được đặt sau ngày tạo review, nên review không thể đánh giá đơn ấy.
- Liên kết chưa xác định: đơn đặt cùng ngày tạo review, hoặc nhiều đơn đều phù hợp về thời gian nhưng chưa biết review thuộc đơn nào.
- Tránh đếm lặp: khi thống kê ở cấp khách hàng, mỗi review_id chỉ được tính một lần.

### Cell code 14: Phân loại và lưu liên kết review

- **Mục đích:** Gắn trạng thái thời gian cho review–đơn và tổng hợp review một lần ở cấp khách hàng.
- **Vì sao:** Liên kết mua sau review cần tách khỏi liên kết hợp lệ hoặc chưa rõ.
- **Giúp ích:** Tạo hai tệp đã xử lý để dùng có kiểm soát ở các bước sau.


In [ ]:
from pathlib import Path
import shutil

review_links_processed = spark.sql("""
    WITH base AS (
        SELECT
            c.customer_unique_id,
            r.review_id,
            r.order_id,
            r.review_score,
            r.review_comment_title,
            r.review_comment_message,
            r.review_creation_date,
            r.review_answer_timestamp,
            o.order_purchase_timestamp,
            o.order_status,
            o.order_delivered_customer_date,
            to_date(o.order_purchase_timestamp) AS ngay_mua,
            to_date(substr(r.review_creation_date, 1, 10)) AS ngay_tao_review
        FROM reviews r
        LEFT JOIN orders o ON r.order_id = o.order_id
        LEFT JOIN customers c ON o.customer_id = c.customer_id
    ),
    counted AS (
        SELECT
            *,
            SUM(
                CASE
                    WHEN ngay_mua <= ngay_tao_review THEN 1
                    ELSE 0
                END
            ) OVER (
                PARTITION BY customer_unique_id, review_id
            ) AS so_don_co_the_lien_ket
        FROM base
    )
    SELECT
        customer_unique_id,
        review_id,
        order_id,
        review_score,
        review_comment_title,
        review_comment_message,
        review_creation_date,
        review_answer_timestamp,
        order_purchase_timestamp,
        order_status,
        order_delivered_customer_date,
        so_don_co_the_lien_ket,
        CASE
            WHEN ngay_mua IS NULL OR ngay_tao_review IS NULL
                THEN 'CHUA_XAC_DINH_THIEU_THOI_GIAN'
            WHEN ngay_mua > ngay_tao_review
                THEN 'LOAI_LIEN_KET_REVIEW_DON'
            WHEN ngay_mua = ngay_tao_review
                THEN 'CHUA_XAC_DINH_CUNG_NGAY'
            WHEN so_don_co_the_lien_ket > 1
                THEN 'CHUA_XAC_DINH_NHIEU_DON'
            ELSE 'HOP_LE_VE_THOI_GIAN'
        END AS trang_thai_lien_ket
    FROM counted
""").cache()

review_links_processed.createOrReplaceTempView("review_links_processed")

reviews_per_customer = spark.sql("""
    SELECT
        customer_unique_id,
        review_id,
        MAX(review_score) AS review_score,
        MAX(review_comment_title) AS review_comment_title,
        MAX(review_comment_message) AS review_comment_message,
        MAX(review_creation_date) AS review_creation_date,
        MAX(review_answer_timestamp) AS review_answer_timestamp,
        COUNT(*) AS so_dong_lien_ket_goc,
        COUNT(DISTINCT order_id) AS so_don_lien_ket,
        SUM(
            CASE
                WHEN trang_thai_lien_ket = 'LOAI_LIEN_KET_REVIEW_DON' THEN 1
                ELSE 0
            END
        ) AS so_lien_ket_bi_loai,
        SUM(
            CASE
                WHEN to_date(order_purchase_timestamp)
                     < to_date(substr(review_creation_date, 1, 10)) THEN 1
                ELSE 0
            END
        ) AS so_don_mua_truoc_review,
        CASE
            WHEN SUM(
                CASE
                    WHEN to_date(order_purchase_timestamp)
                         < to_date(substr(review_creation_date, 1, 10)) THEN 1
                    ELSE 0
                END
            ) > 0
            AND MAX(review_answer_timestamp) IS NOT NULL
                THEN 1
            ELSE 0
        END AS co_the_dung_lam_dac_trung
    FROM review_links_processed
    WHERE customer_unique_id IS NOT NULL
      AND review_id IS NOT NULL
    GROUP BY customer_unique_id, review_id
""").cache()

reviews_per_customer.createOrReplaceTempView("reviews_per_customer")

output_dir = Path(customer_path).parent / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

def save_as_csv(df, filename):
    destination = output_dir / filename
    temporary = Path("/content") / f"_temp_{destination.stem}"

    df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(temporary))
    part_file = next(temporary.glob("part-*.csv"))
    shutil.copyfile(part_file, destination)
    shutil.rmtree(temporary)

    return destination

links_file = save_as_csv(
    review_links_processed,
    "olist_review_order_links_audit.csv"
)

customer_reviews_file = save_as_csv(
    reviews_per_customer,
    "olist_reviews_per_customer.csv"
)

print("SỐ LIÊN KẾT THEO TRẠNG THÁI")
review_links_processed.groupBy("trang_thai_lien_ket").count().orderBy(
    "trang_thai_lien_ket"
).show(truncate=False)

print("THỐNG KÊ REVIEW Ở CẤP KHÁCH HÀNG")
reviews_per_customer.selectExpr(
    "COUNT(*) AS so_review_id_khong_dem_lap",
    "SUM(co_the_dung_lam_dac_trung) AS so_review_co_the_dung_lam_dac_trung",
    "SUM(so_lien_ket_bi_loai) AS so_lien_ket_bi_loai"
).show(truncate=False)

print(links_file)
print(customer_reviews_file)

SỐ LIÊN KẾT THEO TRẠNG THÁI
+------------------------+-----+
|trang_thai_lien_ket     |count|
+------------------------+-----+
|CHUA_XAC_DINH_CUNG_NGAY |10   |
|CHUA_XAC_DINH_NHIEU_DON |1529 |
|HOP_LE_VE_THOI_GIAN     |97621|
|LOAI_LIEN_KET_REVIEW_DON|64   |
+------------------------+-----+

THỐNG KÊ REVIEW Ở CẤP KHÁCH HÀNG
+--------------------------+-----------------------------------+-------------------+
|so_review_id_khong_dem_lap|so_review_co_the_dung_lam_dac_trung|so_lien_ket_bi_loai|
+--------------------------+-----------------------------------+-------------------+
|98410                     |98378                              |64                 |
+--------------------------+-----------------------------------+-------------------+

/content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_review_order_links_audit.csv
/content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_reviews_per_customer.csv


Cell code: Trực quan hóa kết quả phân loại trạng thái liên kết và review khách hàng
Mục đích: Sử dụng thư viện hvplot và panel để vẽ biểu đồ cột thể hiện phân bố trạng thái liên kết Review–Đơn hàng và các chỉ số thống kê ở cấp khách hàng.

Vì sao: Đánh giá trực quan tỷ lệ dữ liệu hợp lệ, dữ liệu nghi ngờ hoặc bị loại bỏ sau khi xử lý.

Giúp ích: Cung cấp cái nhìn tổng quan nhanh chóng về chất lượng dữ liệu review trước khi chuyển sang bước Feature Engineering.

In [ ]:
%pip -q install hvplot panel

import pandas as pd
import hvplot.pandas
import panel as pn
from IPython.display import display

pn.extension()

def black_chart_text(plot, element):
    figure = plot.state
    figure.background_fill_color = "white"
    figure.border_fill_color = "white"
    figure.title.text_color = "black"

    for axis in list(figure.xaxis) + list(figure.yaxis):
        axis.axis_label_text_color = "black"
        axis.major_label_text_color = "black"

review_link_status_pd = (
    review_links_processed
    .groupBy("trang_thai_lien_ket")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
    .rename(columns={
        "trang_thai_lien_ket": "Trạng thái liên kết",
        "count": "Số liên kết"
    })
)

display(
    review_link_status_pd.hvplot.bar(
        x="Trạng thái liên kết",
        y="Số liên kết",
        width=1000,
        height=430,
        rot=30,
        title="Phân bố trạng thái liên kết review - đơn hàng"
    ).opts(hooks=[black_chart_text])
)

review_customer_pd = (
    reviews_per_customer
    .selectExpr(
        "COUNT(*) AS so_review_id_khong_dem_lap",
        "SUM(co_the_dung_lam_dac_trung) AS so_review_co_the_dung_lam_dac_trung",
        "SUM(so_lien_ket_bi_loai) AS so_lien_ket_bi_loai"
    )
    .toPandas()
    .T
    .reset_index()
)

review_customer_pd.columns = ["Chỉ số", "Số lượng"]

review_customer_pd["Chỉ số"] = review_customer_pd["Chỉ số"].replace({
    "so_review_id_khong_dem_lap": "Review ID không đếm lặp",
    "so_review_co_the_dung_lam_dac_trung": "Review có thể dùng làm đặc trưng",
    "so_lien_ket_bi_loai": "Liên kết bị loại"
})

display(
    review_customer_pd.hvplot.bar(
        x="Chỉ số",
        y="Số lượng",
        width=900,
        height=400,
        rot=25,
        title="Kết quả xử lý review ở cấp khách hàng"
    ).opts(hooks=[black_chart_text])
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 3.3 MB/s eta 0:00:00


/tmp/ipykernel_780/4030145579.py:8: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


:Bars   [Trạng thái liên kết]   (Số liên kết)

:Bars   [Chỉ số]   (Số lượng)

### Cell code 15: Kiểm tra lại hai tệp review đã lưu

- **Mục đích:** Đếm khóa trùng, khóa thiếu và đối chiếu liên kết với bảng review cấp khách.
- **Vì sao:** Tệp mới vẫn cần được xác nhận sau thao tác tổng hợp và xuất CSV.
- **Giúp ích:** Phát hiện sai lệch trước khi đưa review vào feature.


In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

processed_dir = Path(customer_path).parent / "processed"

links = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(str(processed_dir / "olist_review_order_links_audit.csv"))
)

customer_reviews = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(str(processed_dir / "olist_reviews_per_customer.csv"))
)
def duplicate_stats(df, keys):
    groups = df.groupBy(*keys).count().filter(F.col("count") > 1)
    result = groups.agg(
        F.count("*").alias("so_nhom_trung"),
        F.coalesce(
            F.sum(F.col("count") - 1), F.lit(0)
        ).alias("so_dong_du")
    ).first()
    return result["so_nhom_trung"], result["so_dong_du"]

link_pair_groups, link_pair_extra = duplicate_stats(
    links, ["review_id", "order_id"]
)

customer_review_groups, customer_review_extra = duplicate_stats(
    customer_reviews, ["customer_unique_id", "review_id"]
)

global_review_groups, global_review_extra = duplicate_stats(
    customer_reviews, ["review_id"]
)

link_keys = links.select(
    "customer_unique_id", "review_id"
).distinct()

customer_keys = customer_reviews.select(
    "customer_unique_id", "review_id"
).distinct()

checks = [
    ("Số dòng file liên kết", links.count()),
    ("Số dòng file tổng hợp khách hàng", customer_reviews.count()),
    ("Dòng trùng hoàn toàn trong file liên kết", links.count() - links.distinct().count()),
    ("Cặp review_id–order_id bị trùng", link_pair_groups),
    ("Dòng dư của cặp review_id–order_id", link_pair_extra),
    ("Cặp khách hàng–review_id bị trùng trong file tổng hợp", customer_review_groups),
    ("Dòng dư của cặp khách hàng–review_id", customer_review_extra),
    ("review_id lặp giữa các dòng trong file tổng hợp", global_review_groups),
    ("Dòng dư theo review_id trong file tổng hợp", global_review_extra),
    ("Cặp khách hàng–review_id có ở liên kết nhưng thiếu ở tổng hợp",
     link_keys.join(customer_keys, ["customer_unique_id", "review_id"], "left_anti").count()),
    ("Cặp khách hàng–review_id có ở tổng hợp nhưng thiếu ở liên kết",
     customer_keys.join(link_keys, ["customer_unique_id", "review_id"], "left_anti").count()),
    ("Liên kết thiếu review_id hoặc order_id",
     links.filter(F.col("review_id").isNull() | F.col("order_id").isNull()).count()),
    ("Dòng tổng hợp thiếu customer_unique_id hoặc review_id",
     customer_reviews.filter(
         F.col("customer_unique_id").isNull() | F.col("review_id").isNull()
     ).count())
]

spark.createDataFrame(checks, ["kiem_tra", "so_luong"]).show(
    len(checks), truncate=False
)

print("Review ID lặp trong file liên kết theo số đơn gắn với ID:")
links.groupBy("review_id").agg(
    F.countDistinct("order_id").alias("so_don")
).groupBy("so_don").count().orderBy("so_don").show()

+-------------------------------------------------------------+--------+
|kiem_tra                                                     |so_luong|
+-------------------------------------------------------------+--------+
|Số dòng file liên kết                                        |99224   |
|Số dòng file tổng hợp khách hàng                             |98410   |
|Dòng trùng hoàn toàn trong file liên kết                     |0       |
|Cặp review_id–order_id bị trùng                              |0       |
|Dòng dư của cặp review_id–order_id                           |0       |
|Cặp khách hàng–review_id bị trùng trong file tổng hợp        |0       |
|Dòng dư của cặp khách hàng–review_id                         |0       |
|review_id lặp giữa các dòng trong file tổng hợp              |0       |
|Dòng dư theo review_id trong file tổng hợp                   |0       |
|Cặp khách hàng–review_id có ở liên kết nhưng thiếu ở tổng hợp|0       |
|Cặp khách hàng–review_id có ở tổng hợp nhưng thiếu

In [ ]:
review_validation_pd = pd.DataFrame(
    checks,
    columns=["Kiểm tra", "Số lượng"]
)

display(
    review_validation_pd.hvplot.bar(
        x="Kiểm tra",
        y="Số lượng",
        width=1200,
        height=450,
        rot=55,
        title="Kiểm tra chất lượng dữ liệu review sau xử lý"
    ).opts(hooks=[black_chart_text])
)

review_order_dist_pd = (
    links
    .groupBy("review_id")
    .agg(F.countDistinct("order_id").alias("so_don"))
    .groupBy("so_don")
    .count()
    .orderBy("so_don")
    .toPandas()
    .rename(columns={
        "so_don": "Số đơn liên kết",
        "count": "Số review ID"
    })
)

display(
    review_order_dist_pd.hvplot.bar(
        x="Số đơn liên kết",
        y="Số review ID",
        width=750,
        height=400,
        title="Số đơn được liên kết với mỗi review ID"
    ).opts(hooks=[black_chart_text])
)

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

##Kiểm tra bảng orders

### Cell code 16: Kiểm tra khóa bảng orders

- **Mục đích:** Xác nhận order_id duy nhất và customer_id nối được với bảng customers.
- **Vì sao:** Mỗi đơn phải có đúng một khóa và nhận diện được khách hàng.
- **Giúp ích:** Đảm bảo có thể thống kê giao dịch theo customer_unique_id.


In [ ]:
from pyspark.sql import functions as F

orders_key = orders.select(
    F.trim(F.col("order_id")).alias("order_id"),
    F.trim(F.col("customer_id")).alias("customer_id")
)

customers_key = customers.select(
    F.trim(F.col("customer_id")).alias("customer_id"),
    F.trim(F.col("customer_unique_id")).alias("customer_unique_id")
)

orders_key = orders_key.withColumn(
    "order_id", F.when(F.col("order_id") == "", None).otherwise(F.col("order_id"))
).withColumn(
    "customer_id", F.when(F.col("customer_id") == "", None).otherwise(F.col("customer_id"))
)

customers_key = customers_key.withColumn(
    "customer_id", F.when(F.col("customer_id") == "", None).otherwise(F.col("customer_id"))
).withColumn(
    "customer_unique_id",
    F.when(F.col("customer_unique_id") == "", None).otherwise(F.col("customer_unique_id"))
)

duplicate_orders = (
    orders_key
    .filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_customers = (
    customers_key
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

unmatched_orders = (
    orders_key
    .filter(F.col("customer_id").isNotNull())
    .join(
        customers_key.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

matched_missing_unique_id = (
    orders_key
    .filter(F.col("customer_id").isNotNull())
    .join(customers_key, on="customer_id", how="inner")
    .filter(F.col("customer_unique_id").isNull())
)

result = [
    ("Tổng số đơn hàng", orders_key.count()),
    ("Đơn thiếu order_id", orders_key.filter(F.col("order_id").isNull()).count()),
    ("Số order_id bị trùng", duplicate_orders.count()),
    ("Số dòng dư do trùng order_id",
     duplicate_orders.agg(F.coalesce(F.sum(F.col("count") - 1), F.lit(0))).first()[0]),
    ("Đơn thiếu customer_id", orders_key.filter(F.col("customer_id").isNull()).count()),
    ("customer_id trùng trong bảng customers", duplicate_customers.count()),
    ("Đơn không nối được với customers", unmatched_orders.count()),
    ("Đơn nối được nhưng thiếu customer_unique_id", matched_missing_unique_id.count())
]

spark.createDataFrame(result, ["kiem_tra", "so_luong"]).show(
    len(result), truncate=False
)

+-------------------------------------------+--------+
|kiem_tra                                   |so_luong|
+-------------------------------------------+--------+
|Tổng số đơn hàng                           |99441   |
|Đơn thiếu order_id                         |0       |
|Số order_id bị trùng                       |0       |
|Số dòng dư do trùng order_id               |0       |
|Đơn thiếu customer_id                      |0       |
|customer_id trùng trong bảng customers     |0       |
|Đơn không nối được với customers           |0       |
|Đơn nối được nhưng thiếu customer_unique_id|0       |
+-------------------------------------------+--------+



Cell code: Trực quan hóa kết quả kiểm tra chất lượng khóa của bảng orders
Mục đích: Chuyển đổi danh sách kết quả kiểm tra chất lượng khóa (result) thành DataFrame và hiển thị dưới dạng biểu đồ cột tương tác bằng hvplot.

Vì sao: Đánh giá độ tin cậy của các khóa chính và khóa ngoại (như order_id, customer_id) thông qua việc phát hiện các giá trị trùng lặp hoặc bị khuyết.

Giúp ích: Đảm bảo tính toàn vẹn dữ liệu trước khi thực hiện các thao tác kết nối bảng (join) ở những bước tiếp theo.

In [ ]:
orders_key_quality_pd = pd.DataFrame(
    result,
    columns=["Kiểm tra", "Số lượng"]
)

display(
    orders_key_quality_pd.hvplot.bar(
        x="Kiểm tra",
        y="Số lượng",
        width=1100,
        height=430,
        rot=45,
        title="Kiểm tra chất lượng khóa của bảng orders"
    ).opts(hooks=[black_chart_text])
)

##Thống kê bảng order theo trạng thái đơn

### Cell code 17: Phân bố trạng thái đơn hàng

- **Mục đích:** Đếm số đơn và tỷ lệ theo delivered, canceled và các trạng thái khác.
- **Vì sao:** Không phải mọi đơn được đặt đều là một lần mua hoàn tất.
- **Giúp ích:** Có căn cứ chọn loại đơn để xây dựng lịch sử mua.


In [ ]:
from pyspark.sql import functions as F

orders_status = orders.select(
    F.lower(F.trim(F.col("order_status"))).alias("order_status")
).withColumn(
    "order_status",
    F.when(
        F.col("order_status").isNull() | (F.col("order_status") == ""),
        "không rõ"
    ).otherwise(F.col("order_status"))
)

total_orders = orders_status.count()

status_stats = (
    orders_status
    .groupBy("order_status")
    .agg(F.count("*").alias("so_don"))
    .withColumn(
        "ty_le_phan_tram",
        F.round(F.col("so_don") * 100 / F.lit(total_orders), 2)
    )
    .orderBy(F.desc("so_don"))
)

group_stats = (
    orders_status
    .withColumn(
        "nhom_trang_thai",
        F.when(F.col("order_status") == "delivered", "Đã giao")
         .when(F.col("order_status") == "canceled", "Đã hủy")
         .otherwise("Chưa hoàn tất hoặc trạng thái khác")
    )
    .groupBy("nhom_trang_thai")
    .agg(F.count("*").alias("so_don"))
    .withColumn(
        "ty_le_phan_tram",
        F.round(F.col("so_don") * 100 / F.lit(total_orders), 2)
    )
    .orderBy(F.desc("so_don"))
)

print(f"TỔNG SỐ ĐƠN: {total_orders}")
print("THỐNG KÊ THEO TỪNG ORDER_STATUS")
status_stats.show(truncate=False)

print("THỐNG KÊ THEO NHÓM")
group_stats.show(truncate=False)

TỔNG SỐ ĐƠN: 99441
THỐNG KÊ THEO TỪNG ORDER_STATUS
+------------+------+---------------+
|order_status|so_don|ty_le_phan_tram|
+------------+------+---------------+
|delivered   |96478 |97.02          |
|shipped     |1107  |1.11           |
|canceled    |625   |0.63           |
|unavailable |609   |0.61           |
|invoiced    |314   |0.32           |
|processing  |301   |0.3            |
|created     |5     |0.01           |
|approved    |2     |0.0            |
+------------+------+---------------+

THỐNG KÊ THEO NHÓM
+----------------------------------+------+---------------+
|nhom_trang_thai                   |so_don|ty_le_phan_tram|
+----------------------------------+------+---------------+
|Đã giao                           |96478 |97.02          |
|Chưa hoàn tất hoặc trạng thái khác|2338  |2.35           |
|Đã hủy                            |625   |0.63           |
+----------------------------------+------+---------------+



Cell code: Trực quan hóa phân bố số đơn và tỷ lệ đơn theo nhóm trạng thái
Mục đích: Sử dụng hvplot biểu diễn hai biểu đồ cột: một biểu đồ thống kê số lượng đơn hàng chi tiết theo từng order_status và một biểu đồ phản ánh tỷ lệ phần trăm theo nhóm trạng thái đơn hàng đã phân loại (nhom_trang_thai).

Vì sao: Quan sát trực quan cơ cấu đơn hàng trong toàn bộ hệ thống, giúp dễ dàng so sánh quy mô giữa các trạng thái hoàn thành và chưa hoàn thành.

Giúp ích: Hỗ trợ đánh giá tổng thể bức tranh vận hành đơn hàng và cung cấp thông tin đầu vào quan trọng cho các phân tích EDA ở bước sau.

In [ ]:
status_stats_pd = status_stats.toPandas()

display(
    status_stats_pd.hvplot.bar(
        x="order_status",
        y="so_don",
        width=950,
        height=430,
        rot=30,
        title="Phân bố số đơn theo order_status",
        xlabel="Trạng thái đơn",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

group_stats_pd = group_stats.toPandas()

display(
    group_stats_pd.hvplot.bar(
        x="nhom_trang_thai",
        y="ty_le_phan_tram",
        width=800,
        height=400,
        rot=20,
        title="Tỷ lệ đơn theo nhóm trạng thái",
        xlabel="Nhóm trạng thái",
        ylabel="Tỷ lệ (%)"
    ).opts(hooks=[black_chart_text])
)

:Bars   [order_status]   (so_don)

:Bars   [nhom_trang_thai]   (ty_le_phan_tram)

##Quy trình đơn hàng: đặt hàng → duyệt thanh toán → bàn giao cho đơn vị vận chuyển → giao cho khách

### Cell code 18: Rà soát thời gian của đơn

- **Mục đích:** Kiểm tra mốc đặt, duyệt, bàn giao, giao và ngày giao dự kiến theo trạng thái.
- **Vì sao:** Thứ tự thời gian sai hoặc ngày giao thiếu sẽ làm sai đặc trưng giao hàng.
- **Giúp ích:** Biết mốc nào có thể dùng và đơn nào cần đánh cờ.


In [ ]:
from pyspark.sql import functions as F

time_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_time = orders

for name in time_columns:
    orders_time = orders_time.withColumn(name, F.to_timestamp(F.col(name)))

time_checks = {
    "duyet_truoc_khi_dat": (
        F.col("order_approved_at") < F.col("order_purchase_timestamp")
    ),
    "ban_giao_truoc_khi_dat": (
        F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp")
    ),
    "ban_giao_truoc_khi_duyet": (
        F.col("order_delivered_carrier_date") < F.col("order_approved_at")
    ),
    "giao_khach_truoc_khi_dat": (
        F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp")
    ),
    "giao_khach_truoc_khi_ban_giao": (
        F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date")
    )
}

for name, condition in time_checks.items():
    orders_time = orders_time.withColumn(
        name, F.when(condition, 1).otherwise(0)
    )

orders_time = orders_time.withColumn(
    "co_sai_thu_tu",
    F.when(
        sum(F.col(name) for name in time_checks) > 0, 1
    ).otherwise(0)
)

missing_by_status = (
    orders_time
    .groupBy("order_status")
    .agg(
        F.count("*").alias("so_don"),
        *[
            F.sum(
                F.when(F.col(name).isNull(), 1).otherwise(0)
            ).alias(f"thieu_{name}")
            for name in time_columns
        ],
        F.sum("co_sai_thu_tu").alias("so_don_sai_thu_tu")
    )
    .orderBy(F.desc("so_don"))
)

order_check_summary = orders_time.agg(
    F.count("*").alias("tong_so_don"),
    *[
        F.sum(name).alias(name)
        for name in time_checks
    ],
    F.sum("co_sai_thu_tu").alias("so_don_sai_it_nhat_mot_thu_tu"),
    F.sum(
        F.when(
            (F.col("order_status") == "delivered") &
            F.col("order_delivered_customer_date").isNull(),
            1
        ).otherwise(0)
    ).alias("don_delivered_thieu_ngay_giao")
)

delivery_vs_estimate = (
    orders_time
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "chenh_lech_ngay",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_estimated_delivery_date")
        )
    )
    .agg(
        F.sum(F.when(F.col("chenh_lech_ngay") < 0, 1).otherwise(0))
         .alias("giao_som"),
        F.sum(F.when(F.col("chenh_lech_ngay") == 0, 1).otherwise(0))
         .alias("giao_dung_ngay"),
        F.sum(F.when(F.col("chenh_lech_ngay") > 0, 1).otherwise(0))
         .alias("giao_tre"),
        F.sum(F.when(F.col("chenh_lech_ngay").isNull(), 1).otherwise(0))
         .alias("khong_du_moc_de_so_sanh")
    )
)

print("MỐC THỜI GIAN THIẾU THEO TRẠNG THÁI")
missing_by_status.show(truncate=False)

print("CÁC TRƯỜNG HỢP SAI THỨ TỰ THỜI GIAN")
order_check_summary.show(truncate=False)

print("SO SÁNH NGÀY GIAO THỰC TẾ VỚI NGÀY DỰ KIẾN")
delivery_vs_estimate.show(truncate=False)

print("MẪU ĐƠN CÓ MỐC THỜI GIAN SAI THỨ TỰ")
orders_time.filter(F.col("co_sai_thu_tu") == 1).select(
    "order_id",
    "order_status",
    *time_columns,
    *time_checks.keys()
).show(orders_time.count(), truncate=False)

MỐC THỜI GIAN THIẾU THEO TRẠNG THÁI
+------------+------+------------------------------+-----------------------+----------------------------------+-----------------------------------+-----------------------------------+-----------------+
|order_status|so_don|thieu_order_purchase_timestamp|thieu_order_approved_at|thieu_order_delivered_carrier_date|thieu_order_delivered_customer_date|thieu_order_estimated_delivery_date|so_don_sai_thu_tu|
+------------+------+------------------------------+-----------------------+----------------------------------+-----------------------------------+-----------------------------------+-----------------+
|delivered   |96478 |0                             |14                     |2                                 |8                                  |0                                  |1373             |
|shipped     |1107  |0                             |0                      |0                                 |1107                               |0        

Cell code : Trực quan hóa các bất thường về mốc thời gian và kết quả giao hàng
Mục đích: Sử dụng hvplot biểu diễn các biểu đồ cột (stacked bar & bar charts) thể hiện tình trạng thiếu mốc thời gian, sai thứ tự thời gian theo trạng thái đơn, thống kê các loại lỗi bất thường logic và so sánh ngày giao thực tế so với dự kiến.

Vì sao: Giúp phát hiện trực quan các nhóm đơn hàng khuyết thông tin thời gian hoặc có tiến trình vận chuyển bất hợp lý (như giao hàng trước khi duyệt đơn).

Giúp ích: Cung cấp cơ sở thực tế để thiết lập các quy tắc lọc dữ liệu rác và gắn cờ chất lượng (quality flags) chuẩn xác hơn.

In [ ]:
missing_time_pd = missing_by_status.toPandas()

missing_columns_plot = [
    "thieu_order_approved_at",
    "thieu_order_delivered_carrier_date",
    "thieu_order_delivered_customer_date",
    "so_don_sai_thu_tu"
]

display(
    missing_time_pd.hvplot.bar(
        x="order_status",
        y=missing_columns_plot,
        stacked=True,
        width=1100,
        height=450,
        rot=30,
        title="Thiếu mốc thời gian và sai thứ tự theo trạng thái đơn",
        xlabel="Trạng thái đơn",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

order_time_error_pd = (
    order_check_summary
    .toPandas()
    .T
    .reset_index()
)

order_time_error_pd.columns = ["Loại kiểm tra", "Số lượng"]

order_time_error_pd = order_time_error_pd[
    order_time_error_pd["Loại kiểm tra"] != "tong_so_don"
]

display(
    order_time_error_pd.hvplot.bar(
        x="Loại kiểm tra",
        y="Số lượng",
        width=1100,
        height=450,
        rot=45,
        title="Các loại bất thường về thứ tự thời gian"
    ).opts(hooks=[black_chart_text])
)

delivery_estimate_pd = (
    delivery_vs_estimate
    .toPandas()
    .T
    .reset_index()
)

delivery_estimate_pd.columns = ["Kết quả giao hàng", "Số đơn"]

display(
    delivery_estimate_pd.hvplot.bar(
        x="Kết quả giao hàng",
        y="Số đơn",
        width=850,
        height=400,
        rot=25,
        title="Giao thực tế so với ngày dự kiến"
    ).opts(hooks=[black_chart_text])
)

:Bars   [order_status,Variable]   (value)

:Bars   [Loại kiểm tra]   (Số lượng)

:Bars   [Kết quả giao hàng]   (Số đơn)

##Nhận xét:
- Trong 96.478 đơn delivered, có 8 đơn thiếu ngày giao thực tế, 14 đơn thiếu ngày duyệt và 2 đơn thiếu ngày bàn giao.
- 1.382 đơn bị gắn ít nhất một cờ sai thứ tự. 166 đơn có ngày bàn giao trước ngày đặt và 23 đơn có ngày giao khách trước ngày bàn giao.
- 1.359 đơn có ngày bàn giao trước ngày duyệt.
- So với ngày dự kiến, 6.534 đơn giao trễ.

### Cell code 19: Xem đơn có mốc thời gian bất thường

- **Mục đích:** Hiển thị đơn bàn giao trước khi đặt, giao trước bàn giao và delivered thiếu ngày giao.
- **Vì sao:** Thống kê tổng số chưa cho biết độ lệch cụ thể của từng đơn.
- **Giúp ích:** Có bằng chứng để đặt quy tắc xử lý, không xóa đơn vội.


In [ ]:
from pyspark.sql import functions as F

orders_detail = (
    orders
    .withColumn("ngay_dat", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("ngay_duyet", F.to_timestamp("order_approved_at"))
    .withColumn("ngay_ban_giao", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("ngay_giao_khach", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("ngay_giao_du_kien", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn(
        "phut_ban_giao_so_voi_dat",
        F.round(
            (F.unix_timestamp("ngay_ban_giao") - F.unix_timestamp("ngay_dat")) / 60,
            1
        )
    )
    .withColumn(
        "phut_giao_khach_so_voi_ban_giao",
        F.round(
            (F.unix_timestamp("ngay_giao_khach") - F.unix_timestamp("ngay_ban_giao")) / 60,
            1
        )
    )
)

ban_giao_truoc_dat = (
    orders_detail
    .filter(F.col("ngay_ban_giao") < F.col("ngay_dat"))
    .select(
        "order_id", "customer_id", "order_status",
        "ngay_dat", "ngay_duyet", "ngay_ban_giao",
        "ngay_giao_khach", "ngay_giao_du_kien",
        "phut_ban_giao_so_voi_dat"
    )
    .orderBy("phut_ban_giao_so_voi_dat")
)

giao_truoc_ban_giao = (
    orders_detail
    .filter(F.col("ngay_giao_khach") < F.col("ngay_ban_giao"))
    .select(
        "order_id", "customer_id", "order_status",
        "ngay_dat", "ngay_duyet", "ngay_ban_giao",
        "ngay_giao_khach", "ngay_giao_du_kien",
        "phut_giao_khach_so_voi_ban_giao"
    )
    .orderBy("phut_giao_khach_so_voi_ban_giao")
)

delivered_thieu_ngay_giao = (
    orders_detail
    .filter(
        (F.col("order_status") == "delivered") &
        F.col("ngay_giao_khach").isNull()
    )
    .select(
        "order_id", "customer_id", "order_status",
        "ngay_dat", "ngay_duyet", "ngay_ban_giao",
        "ngay_giao_khach", "ngay_giao_du_kien"
    )
    .orderBy("ngay_dat")
)

print(f"BÀN GIAO TRƯỚC KHI ĐẶT: {ban_giao_truoc_dat.count()} ĐƠN")
ban_giao_truoc_dat.show(ban_giao_truoc_dat.count(), truncate=False)

print(f"GIAO CHO KHÁCH TRƯỚC KHI BÀN GIAO: {giao_truoc_ban_giao.count()} ĐƠN")
giao_truoc_ban_giao.show(giao_truoc_ban_giao.count(), truncate=False)

print(f"TRẠNG THÁI DELIVERED NHƯNG THIẾU NGÀY GIAO: {delivered_thieu_ngay_giao.count()} ĐƠN")
delivered_thieu_ngay_giao.show(delivered_thieu_ngay_giao.count(), truncate=False)

BÀN GIAO TRƯỚC KHI ĐẶT: 166 ĐƠN
+--------------------------------+--------------------------------+------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------------+
|order_id                        |customer_id                     |order_status|ngay_dat           |ngay_duyet         |ngay_ban_giao      |ngay_giao_khach    |ngay_giao_du_kien  |phut_ban_giao_so_voi_dat|
+--------------------------------+--------------------------------+------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------------+
|7c48bb55e8e4f7e56d412e9653db37bc|34ef6181341eb36c47fd601c46878f00|delivered   |2018-07-16 18:40:53|2018-07-16 18:50:22|2018-01-26 13:35:00|2018-07-23 20:04:45|2018-08-07 00:00:00|-246545.9               |
|4021cd7611d6d9ce5ffcd24817fc374f|ac27243bffbc9c58eb62adfb58c71488|delivered   |2018-08-18 11:49:40|2018-08-18 12:10:30|2018-08-14 06:22:00|2018

##Nhận xét:
- 166 đơn có thời gian bàn giao trước lúc đặt; phần lớn lệch trong cùng ngày, nhưng có trường hợp lệch nhiều ngày, thậm chí nhiều tháng.
- 23 đơn ghi giao cho khách trước khi bàn giao cho đơn vị vận chuyển. Cả 23 vẫn mang trạng thái delivereddelivered.
- 8 đơn delivered thiếu ngày giao thực tế; một trong số đó còn thiếu cả ngày bàn giao.

### Cell code 20: Đo mức độ và thời điểm bất thường

- **Mục đích:** Nhóm lỗi theo khoảng lệch, tháng đặt và trạng thái đơn.
- **Vì sao:** Lệch vài phút và lệch nhiều ngày có mức ảnh hưởng khác nhau.
- **Giúp ích:** Nhìn rõ mức độ lỗi trước khi dùng dữ liệu giao hàng.


In [ ]:
from pyspark.sql import functions as F

orders_check = (
    orders
    .withColumn("ngay_dat", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("ngay_duyet", F.to_timestamp("order_approved_at"))
    .withColumn("ngay_ban_giao", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("ngay_giao_khach", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("thang_dat", F.date_format("ngay_dat", "yyyy-MM"))
)

pairs = [
    ("Bàn giao trước khi đặt", "ngay_dat", "ngay_ban_giao"),
    ("Bàn giao trước khi duyệt", "ngay_duyet", "ngay_ban_giao"),
    ("Giao khách trước khi bàn giao", "ngay_ban_giao", "ngay_giao_khach")
]

anomalies = None

for label, expected_later, recorded_earlier in pairs:
    current = (
        orders_check
        .filter(F.col(recorded_earlier) < F.col(expected_later))
        .select(
            "order_id",
            "order_status",
            "thang_dat",
            F.lit(label).alias("loai_bat_thuong"),
            F.round(
                (
                    F.unix_timestamp(expected_later)
                    - F.unix_timestamp(recorded_earlier)
                ) / 60,
                1
            ).alias("lech_phut")
        )
    )
    anomalies = current if anomalies is None else anomalies.unionByName(current)

anomalies = anomalies.withColumn(
    "khoang_lech",
    F.when(F.col("lech_phut") <= 30, "01. Đến 30 phút")
     .when(F.col("lech_phut") <= 60, "02. Trên 30 phút đến 1 giờ")
     .when(F.col("lech_phut") <= 120, "03. Trên 1 đến 2 giờ")
     .when(F.col("lech_phut") <= 1440, "04. Trên 2 giờ đến 1 ngày")
     .when(F.col("lech_phut") <= 10080, "05. Trên 1 đến 7 ngày")
     .otherwise("06. Trên 7 ngày")
)

print("SỐ ĐƠN THEO LOẠI BẤT THƯỜNG VÀ MỨC ĐỘ LỆCH")
anomalies.groupBy("loai_bat_thuong", "khoang_lech").agg(
    F.countDistinct("order_id").alias("so_don")
).orderBy("loai_bat_thuong", "khoang_lech").show(anomalies.count(), truncate=False)

print("SỐ ĐƠN THEO THÁNG ĐẶT VÀ LOẠI BẤT THƯỜNG")
anomalies.groupBy("thang_dat", "loai_bat_thuong").agg(
    F.countDistinct("order_id").alias("so_don")
).orderBy("thang_dat", "loai_bat_thuong").show(anomalies.count(), truncate=False)

print("SỐ ĐƠN THEO TRẠNG THÁI VÀ LOẠI BẤT THƯỜNG")
anomalies.groupBy("loai_bat_thuong", "order_status").agg(
    F.countDistinct("order_id").alias("so_don")
).orderBy("loai_bat_thuong", "order_status").show(anomalies.count(), truncate=False)

print("CÁC TRƯỜNG HỢP LỆCH TRÊN 7 NGÀY")
anomalies.filter(
    F.col("lech_phut") > 10080
).orderBy(F.desc("lech_phut")).show(anomalies.count(), truncate=False)

print("ĐƠN DELIVERED THIẾU NGÀY GIAO THEO THÁNG ĐẶT")
orders_check.filter(
    (F.col("order_status") == "delivered") &
    F.col("ngay_giao_khach").isNull()
).groupBy("thang_dat").agg(
    F.count("*").alias("so_don")
).orderBy("thang_dat").show(orders_check.count(), truncate=False)

SỐ ĐƠN THEO LOẠI BẤT THƯỜNG VÀ MỨC ĐỘ LỆCH
+-----------------------------+--------------------------+------+
|loai_bat_thuong              |khoang_lech               |so_don|
+-----------------------------+--------------------------+------+
|Bàn giao trước khi duyệt     |01. Đến 30 phút           |125   |
|Bàn giao trước khi duyệt     |02. Trên 30 phút đến 1 giờ|117   |
|Bàn giao trước khi duyệt     |03. Trên 1 đến 2 giờ      |193   |
|Bàn giao trước khi duyệt     |04. Trên 2 giờ đến 1 ngày |466   |
|Bàn giao trước khi duyệt     |05. Trên 1 đến 7 ngày     |444   |
|Bàn giao trước khi duyệt     |06. Trên 7 ngày           |14    |
|Bàn giao trước khi đặt       |01. Đến 30 phút           |72    |
|Bàn giao trước khi đặt       |02. Trên 30 phút đến 1 giờ|49    |
|Bàn giao trước khi đặt       |03. Trên 1 đến 2 giờ      |39    |
|Bàn giao trước khi đặt       |04. Trên 2 giờ đến 1 ngày |4     |
|Bàn giao trước khi đặt       |05. Trên 1 đến 7 ngày     |1     |
|Bàn giao trước khi đặt       |06

Cell code: Trực quan hóa mức độ lệch và xu hướng bất thường thời gian
Mục đích: Sử dụng hvplot để vẽ biểu đồ cột chồng (stacked bar chart) thể hiện phân bố khoảng lệch thời gian theo từng loại bất thường và biểu đồ đường (line chart) theo dõi xu hướng phát sinh bất thường qua các tháng.

Vì sao: Đánh giá mức độ nghiêm trọng của các sai lệch thời gian (lệch vài giờ vs. lệch nhiều ngày) và kiểm tra xem lỗi dữ liệu xảy ra mang tính hệ thống hay bùng phát theo giai đoạn.

Giúp ích: Cung cấp góc nhìn toàn diện về chất lượng dữ liệu chuỗi thời gian, làm cơ sở ra quyết định xử lý loại bỏ hay gắn cờ dữ liệu nhiễu.

In [ ]:
anomaly_range_pd = (
    anomalies
    .groupBy("loai_bat_thuong", "khoang_lech")
    .agg(F.countDistinct("order_id").alias("so_don"))
    .orderBy("khoang_lech")
    .toPandas()
)

display(
    anomaly_range_pd.hvplot.bar(
        x="khoang_lech",
        y="so_don",
        by="loai_bat_thuong",
        stacked=True,
        width=1100,
        height=470,
        rot=35,
        title="Mức độ lệch thời gian theo loại bất thường",
        xlabel="Khoảng lệch",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

anomaly_month_pd = (
    anomalies
    .groupBy("thang_dat", "loai_bat_thuong")
    .agg(F.countDistinct("order_id").alias("so_don"))
    .orderBy("thang_dat")
    .toPandas()
)

display(
    anomaly_month_pd.hvplot.line(
        x="thang_dat",
        y="so_don",
        by="loai_bat_thuong",
        width=1100,
        height=450,
        rot=45,
        marker="circle",
        title="Bất thường thời gian theo tháng",
        xlabel="Tháng đặt",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

:Bars   [khoang_lech,loai_bat_thuong]   (so_don)

:NdOverlay   [loai_bat_thuong]
   :Curve   [thang_dat]   (so_don)

##Nhận xét:
- 160/166 đơn bàn giao trước khi đặt lệch không quá 2 giờ; 6 đơn còn lại lệch lâu hơn, có một trường hợp lệch nhiều tháng. Chưa đủ cơ sở kết luận nguyên nhân là múi giờ.
- 1.359 đơn bàn giao trước khi duyệt có mức lệch rất khác nhau: 901 đơn không quá 1 ngày, nhưng 14 đơn trên 7 ngày.
- 23 đơn giao khách trước khi bàn giao đều mang trạng thái delivered; 16 đơn lệch trên 1 ngày.
- 8 đơn delivered thiếu ngày giao.

### Cell code 21: Gắn cờ và lưu bảng orders

- **Mục đích:** Giữ đơn, gắn cờ từng lỗi thời gian và xác định khi nào tính được feature giao hàng.
- **Vì sao:** Một mốc sai không nên khiến toàn bộ lịch sử mua của khách bị loại.
- **Giúp ích:** Tệp orders đã xử lý giữ được dấu vết chất lượng dữ liệu.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
from pyspark.sql import functions as F

processed_dir = Path(customer_path).parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "olist_orders_prepared.csv"

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_prepared = orders

for name in date_columns:
    orders_prepared = orders_prepared.withColumn(
        name, F.to_timestamp(F.col(name))
    )

purchase = F.col("order_purchase_timestamp")
approved = F.col("order_approved_at")
carrier = F.col("order_delivered_carrier_date")
delivered = F.col("order_delivered_customer_date")
estimated = F.col("order_estimated_delivery_date")

flags = {
    "co_ban_giao_truoc_dat": carrier < purchase,
    "co_ban_giao_truoc_duyet": carrier < approved,
    "co_giao_truoc_ban_giao": delivered < carrier,
    "co_thieu_ngay_giao_delivered": (
        (F.col("order_status") == "delivered") & delivered.isNull()
    ),
    "co_thieu_ngay_ban_giao_delivered": (
        (F.col("order_status") == "delivered") & carrier.isNull()
    ),
    "co_thieu_ngay_duyet_delivered": (
        (F.col("order_status") == "delivered") & approved.isNull()
    )
}

for name, condition in flags.items():
    orders_prepared = orders_prepared.withColumn(
        name, F.when(condition, 1).otherwise(0)
    )

valid_delivery = (
    (F.col("order_status") == "delivered") &
    purchase.isNotNull() &
    approved.isNotNull() &
    carrier.isNotNull() &
    delivered.isNotNull() &
    (purchase <= approved) &
    (approved <= carrier) &
    (carrier <= delivered)
)

orders_prepared = (
    orders_prepared
    .withColumn(
        "co_the_tinh_dac_trung_giao_hang",
        F.when(valid_delivery, 1).otherwise(0)
    )
    .withColumn(
        "so_ngay_tu_dat_den_giao",
        F.when(
            valid_delivery,
            F.round(
                (F.unix_timestamp(delivered) - F.unix_timestamp(purchase))
                / 86400,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "so_ngay_tu_ban_giao_den_giao",
        F.when(
            valid_delivery,
            F.round(
                (F.unix_timestamp(delivered) - F.unix_timestamp(carrier))
                / 86400,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "giao_tre_so_voi_du_kien",
        F.when(
            valid_delivery & estimated.isNotNull(),
            F.when(F.to_date(delivered) > F.to_date(estimated), 1).otherwise(0)
        ).otherwise(F.lit(None).cast("int"))
    )
)

original_count = orders.count()
prepared_count = orders_prepared.count()

if original_count != prepared_count:
    raise ValueError(
        f"Số dòng thay đổi: gốc={original_count}, xử lý={prepared_count}"
    )

with TemporaryDirectory(prefix="olist_orders_prepared_") as temporary:
    spark_dir = Path(temporary) / "spark_csv"

    (
        orders_prepared
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(str(spark_dir))
    )

    part_file = next(spark_dir.glob("part-*.csv"))
    shutil.copy2(part_file, output_path)

saved_count = spark.read.option("header", True).csv(str(output_path)).count()

if saved_count != original_count:
    raise ValueError(
        f"File đã lưu có {saved_count} dòng, dữ liệu gốc có {original_count} dòng"
    )

print(f"Đã lưu {saved_count} đơn vào: {output_path}")

orders_prepared.agg(
    F.count("*").alias("tong_so_don"),
    *[F.sum(name).alias(name) for name in flags],
    F.sum("co_the_tinh_dac_trung_giao_hang").alias(
        "so_don_co_the_tinh_giao_hang"
    )
).show(truncate=False)

Đã lưu 99441 đơn vào: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_orders_prepared.csv
+-----------+---------------------+-----------------------+----------------------+----------------------------+--------------------------------+-----------------------------+----------------------------+
|tong_so_don|co_ban_giao_truoc_dat|co_ban_giao_truoc_duyet|co_giao_truoc_ban_giao|co_thieu_ngay_giao_delivered|co_thieu_ngay_ban_giao_delivered|co_thieu_ngay_duyet_delivered|so_don_co_the_tinh_giao_hang|
+-----------+---------------------+-----------------------+----------------------+----------------------------+--------------------------------+-----------------------------+----------------------------+
|99441      |166                  |1359                   |23                    |8                           |2                               |14                           |95082                       |
+-----------+---------------------+-----------------------+---------------

Cell code: Trực quan hóa kết quả gắn cờ chất lượng bảng orders
Mục đích: Tổng hợp số lượng các cờ chất lượng (flags) và số đơn hàng đủ điều kiện tính toán đặc trưng giao hàng từ dataframe orders_prepared, sau đó biểu diễn bằng biểu đồ cột tương tác hvplot.

Vì sao: Đánh giá tổng quan số lượng đơn hàng vi phạm các quy tắc logic thời gian (bàn giao/giao hàng sai thứ tự, thiếu thông tin mốc thời gian) và tỷ lệ đơn hàng dữ liệu chuẩn.

Giúp ích: Xác nhận tính chính xác của bước gắn cờ dữ liệu (quality flagging) và xác định quy mô tập dữ liệu hợp lệ sẵn sàng phục vụ tính toán đặc trưng (feature engineering).

In [ ]:
orders_flags_pd = (
    orders_prepared
    .agg(
        *[F.sum(name).alias(name) for name in flags],
        F.sum("co_the_tinh_dac_trung_giao_hang").alias(
            "so_don_co_the_tinh_giao_hang"
        )
    )
    .toPandas()
    .T
    .reset_index()
)

orders_flags_pd.columns = ["Chỉ số", "Số đơn"]

orders_flags_pd["Chỉ số"] = orders_flags_pd["Chỉ số"].replace({
    "co_ban_giao_truoc_dat": "Bàn giao trước khi đặt",
    "co_ban_giao_truoc_duyet": "Bàn giao trước khi duyệt",
    "co_giao_truoc_ban_giao": "Giao trước khi bàn giao",
    "co_thieu_ngay_giao_delivered": "Delivered thiếu ngày giao",
    "co_thieu_ngay_ban_giao_delivered": "Delivered thiếu ngày bàn giao",
    "co_thieu_ngay_duyet_delivered": "Delivered thiếu ngày duyệt",
    "so_don_co_the_tinh_giao_hang": "Có thể tính đặc trưng giao hàng"
})

display(
    orders_flags_pd.hvplot.bar(
        x="Chỉ số",
        y="Số đơn",
        width=1100,
        height=450,
        rot=45,
        title="Kết quả gắn cờ chất lượng bảng orders"
    ).opts(hooks=[black_chart_text])
)

:Bars   [Chỉ số]   (Số đơn)

### Cell code 22: Xác định phạm vi thời gian quan sát

- **Mục đích:** Thống kê ngày mua theo trạng thái/tháng và mốc dự đoán muộn nhất cho 90, 120, 180 ngày.
- **Vì sao:** Gán nhãn churn chỉ đúng khi có đủ thời gian nhìn về tương lai.
- **Giúp ích:** Giảm nguy cơ gán nhầm khách mua gần cuối dữ liệu là churn.


In [ ]:
from pyspark.sql import functions as F

orders_period = (
    orders_prepared
    .select(
        "order_id",
        "order_status",
        F.to_date("order_purchase_timestamp").alias("ngay_dat")
    )
    .withColumn("thang_dat", F.date_format("ngay_dat", "yyyy-MM"))
)

print("PHẠM VI NGÀY ĐẶT HÀNG THEO TRẠNG THÁI")
period_by_status = (
    orders_period
    .groupBy("order_status")
    .agg(
        F.count("*").alias("so_don"),
        F.sum(
            F.when(F.col("ngay_dat").isNull(), 1).otherwise(0)
        ).alias("thieu_ngay_dat"),
        F.min("ngay_dat").alias("ngay_som_nhat"),
        F.max("ngay_dat").alias("ngay_muon_nhat")
    )
    .orderBy(F.desc("so_don"))
)
period_by_status.show(truncate=False)

print("SỐ ĐƠN THEO THÁNG VÀ TRẠNG THÁI")
monthly_orders = (
    orders_period
    .groupBy("thang_dat", "order_status")
    .agg(F.count("*").alias("so_don"))
    .orderBy("thang_dat", "order_status")
)
monthly_orders.show(monthly_orders.count(), truncate=False)

last_dates = orders_period.agg(
    F.max("ngay_dat").alias("ngay_cuoi_tat_ca"),
    F.max(
        F.when(F.col("order_status") == "delivered", F.col("ngay_dat"))
    ).alias("ngay_cuoi_delivered")
).first()

end_dates = spark.createDataFrame(
    [
        ("Tất cả trạng thái", last_dates["ngay_cuoi_tat_ca"]),
        ("Chỉ đơn delivered", last_dates["ngay_cuoi_delivered"])
    ],
    ["pham_vi_don", "ngay_cuoi_du_lieu"]
)

thresholds = spark.createDataFrame(
    [(90,), (120,), (180,)],
    ["nguong_khong_mua_ngay"]
)

print("MỐC DỰ ĐOÁN MUỘN NHẤT THEO TỪNG NGƯỠNG")

(
    end_dates
    .crossJoin(thresholds)
    .withColumn(
        "nguong_khong_mua_ngay",
        F.col("nguong_khong_mua_ngay").cast("int")
    )
    .withColumn(
        "moc_du_doan_muon_nhat",
        F.expr("date_sub(ngay_cuoi_du_lieu, nguong_khong_mua_ngay)")
    )
    .orderBy("pham_vi_don", "nguong_khong_mua_ngay")
    .show(truncate=False)
)

PHẠM VI NGÀY ĐẶT HÀNG THEO TRẠNG THÁI
+------------+------+--------------+-------------+--------------+
|order_status|so_don|thieu_ngay_dat|ngay_som_nhat|ngay_muon_nhat|
+------------+------+--------------+-------------+--------------+
|delivered   |96478 |0             |2016-09-15   |2018-08-29    |
|shipped     |1107  |0             |2016-09-04   |2018-09-03    |
|canceled    |625   |0             |2016-09-05   |2018-10-17    |
|unavailable |609   |0             |2016-10-05   |2018-08-21    |
|invoiced    |314   |0             |2016-10-04   |2018-08-14    |
|processing  |301   |0             |2016-10-05   |2018-07-23    |
|created     |5     |0             |2017-11-06   |2018-02-09    |
|approved    |2     |0             |2017-02-06   |2017-04-25    |
+------------+------+--------------+-------------+--------------+

SỐ ĐƠN THEO THÁNG VÀ TRẠNG THÁI
+---------+------------+------+
|thang_dat|order_status|so_don|
+---------+------------+------+
|2016-09  |canceled    |2     |
|2016-09 

Cell code: Trực quan hóa xu hướng đơn hàng theo tháng và phân bố theo trạng thái
Mục đích: Sử dụng hvplot để vẽ biểu đồ đường xu hướng số lượng đơn hàng theo tháng (thang_dat) phân theo từng trạng thái và biểu đồ cột tổng quan số lượng đơn hàng theo order_status trong toàn bộ thời gian quan sát.

Vì sao: Theo dõi sự biến động của lượng đơn hàng qua các thời kỳ, đồng thời so sánh quy mô tổng thể giữa các trạng thái đơn hàng (đã giao, bị hủy, đang xử lý,...).

Giúp ích: Cung cấp bức tranh toàn cảnh về tiến trình vận động đơn hàng theo chuỗi thời gian, giúp phát hiện các giai đoạn bất thường và đánh giá mức độ đại diện của dữ liệu.

In [ ]:
monthly_orders_pd = monthly_orders.toPandas()

display(
    monthly_orders_pd.hvplot.line(
        x="thang_dat",
        y="so_don",
        by="order_status",
        width=1200,
        height=480,
        rot=45,
        marker="circle",
        title="Số đơn theo tháng và trạng thái",
        xlabel="Tháng",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

period_status_pd = period_by_status.toPandas()

display(
    period_status_pd.hvplot.bar(
        x="order_status",
        y="so_don",
        width=950,
        height=420,
        rot=30,
        title="Tổng số đơn theo trạng thái trong toàn bộ thời gian quan sát",
        xlabel="Trạng thái",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

:NdOverlay   [order_status]
   :Curve   [thang_dat]   (so_don)

:Bars   [order_status]   (so_don)

##Nhận xét:
- Ngày đặt không bị thiếu ở bất kỳ trạng thái nào. Đơn delivered có dữ liệu từ 15/09/2016 đến 29/08/2018, tập trung chủ yếu từ năm 2017 đến tháng 8/2018.
- Sau tháng 8/2018 không còn đơn delivered; tháng 9–10 chỉ có rất ít đơn shipped hay canceled.

### Cell code 23: Đối chiếu dấu hiệu giao dịch theo trạng thái

- **Mục đích:** Xem từng nhóm đơn có dòng sản phẩm, thanh toán, ngày duyệt và ngày giao không.
- **Vì sao:** Có thanh toán chưa chắc nghĩa là đơn đã hoàn tất.
- **Giúp ích:** Củng cố quy tắc dùng đơn delivered làm lần mua hoàn tất.


In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

data_dir = Path(customer_path).parent

order_items = (
    spark.read
    .option("header", True)
    .csv(str(data_dir / "olist_order_items_dataset.csv"))
)

payments = (
    spark.read
    .option("header", True)
    .csv(str(data_dir / "olist_order_payments_dataset.csv"))
)

items_per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.count("*").alias("so_dong_san_pham"))
)

payments_per_order = (
    payments
    .groupBy("order_id")
    .agg(
        F.count("*").alias("so_dong_thanh_toan"),
        F.sum(
            F.col("payment_value").cast("double")
        ).alias("tong_gia_tri_thanh_toan")
    )
)

orders_purchase_check = (
    orders_prepared
    .select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date"
    )
    .join(items_per_order, on="order_id", how="left")
    .join(payments_per_order, on="order_id", how="left")
    .withColumn(
        "co_san_pham",
        F.when(F.col("so_dong_san_pham").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "co_ban_ghi_thanh_toan",
        F.when(F.col("so_dong_thanh_toan").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "co_ngay_duyet",
        F.when(F.col("order_approved_at").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "co_ngay_giao",
        F.when(F.col("order_delivered_customer_date").isNotNull(), 1).otherwise(0)
    )
)

print("SỐ ĐƠN THEO TRẠNG THÁI VÀ DẤU HIỆU GIAO DỊCH")

status_evidence = (
    orders_purchase_check
    .groupBy("order_status")
    .agg(
        F.count("*").alias("so_don"),
        F.sum("co_san_pham").alias("co_san_pham"),
        F.sum("co_ban_ghi_thanh_toan").alias("co_ban_ghi_thanh_toan"),
        F.sum("co_ngay_duyet").alias("co_ngay_duyet"),
        F.sum("co_ngay_giao").alias("co_ngay_giao"),
        F.sum(
            F.when(
                (F.col("co_san_pham") == 1) &
                (F.col("co_ban_ghi_thanh_toan") == 1),
                1
            ).otherwise(0)
        ).alias("co_ca_san_pham_va_thanh_toan")
    )
    .withColumn(
        "ty_le_co_ngay_giao_pct",
        F.round(F.col("co_ngay_giao") * 100 / F.col("so_don"), 2)
    )
    .orderBy(F.desc("so_don"))
)

status_evidence.show(truncate=False)

print("CHI TIẾT CÁC TỔ HỢP THEO TRẠNG THÁI")

(
    orders_purchase_check
    .groupBy(
        "order_status",
        "co_san_pham",
        "co_ban_ghi_thanh_toan",
        "co_ngay_giao"
    )
    .agg(F.count("*").alias("so_don"))
    .orderBy("order_status", F.desc("so_don"))
    .show(100, truncate=False)
)

print("ĐỐI CHIẾU SỐ DÒNG SAU KHI NỐI")
print(
    "Orders gốc:", orders_prepared.count(),
    "| Sau khi nối:", orders_purchase_check.count()
)

SỐ ĐƠN THEO TRẠNG THÁI VÀ DẤU HIỆU GIAO DỊCH
+------------+------+-----------+---------------------+-------------+------------+----------------------------+----------------------+
|order_status|so_don|co_san_pham|co_ban_ghi_thanh_toan|co_ngay_duyet|co_ngay_giao|co_ca_san_pham_va_thanh_toan|ty_le_co_ngay_giao_pct|
+------------+------+-----------+---------------------+-------------+------------+----------------------------+----------------------+
|delivered   |96478 |96478      |96477                |96464        |96470       |96477                       |99.99                 |
|shipped     |1107  |1106       |1107                 |1107         |0           |1106                        |0.0                   |
|canceled    |625   |461        |625                  |484          |6           |461                         |0.96                  |
|unavailable |609   |6          |609                  |609          |0           |6                           |0.0                   |
|invoiced 

Cell code: Trực quan hóa tỷ lệ dấu hiệu giao dịch theo trạng thái đơn
Mục đích: Tính toán tỷ lệ phần trăm các dấu hiệu giao dịch (có sản phẩm, có bản ghi thanh toán, có ngày duyệt, có ngày giao) và sử dụng hvplot để biểu diễn dưới dạng biểu đồ cột ghép theo từng order_status.

Vì sao: Đánh giá mức độ đầy đủ dữ liệu thực tế phát sinh trong chu trình xử lý của từng trạng thái đơn hàng (ví dụ: đơn bị hủy có phát sinh thanh toán hay không).

Giúp ích: Cung cấp bằng chứng trực quan để xác minh tính logic của dữ liệu giao dịch, hỗ trợ định hình các điều kiện lọc và làm sạch dữ liệu chuẩn xác hơn.

In [ ]:
status_evidence_pd = status_evidence.toPandas()

for col in [
    "co_san_pham",
    "co_ban_ghi_thanh_toan",
    "co_ngay_duyet",
    "co_ngay_giao"
]:
    status_evidence_pd[col + "_pct"] = (
        status_evidence_pd[col]
        / status_evidence_pd["so_don"]
        * 100
    )

display(
    status_evidence_pd.hvplot.bar(
        x="order_status",
        y=[
            "co_san_pham_pct",
            "co_ban_ghi_thanh_toan_pct",
            "co_ngay_duyet_pct",
            "co_ngay_giao_pct"
        ],
        width=1150,
        height=470,
        rot=30,
        title="Tỷ lệ dấu hiệu giao dịch theo trạng thái đơn",
        xlabel="Trạng thái đơn",
        ylabel="Tỷ lệ (%)"
    ).opts(hooks=[black_chart_text])
)

:Bars   [order_status,Variable]   (value)

##Nhận xét:
- 96.478 đơn delivered đều có sản phẩm; 96.477 có bản ghi thanh toán và 96.470 có ngày giao.
- Mọi trạng thái đều có bản ghi thanh toán, kể cả canceled và unavailable. Vì vậy, chỉ thấy đơn trong bảng payments thì không thể kết luận đơn đã mua thành công.
- Có 6 đơn canceled nhưng lại có ngày giao.
-

### Cell code 24: Kiểm tra đơn canceled vẫn có ngày giao

- **Mục đích:** Xem chi tiết sáu đơn hủy nhưng có ngày giao, sản phẩm và thanh toán.
- **Vì sao:** Trạng thái và dấu hiệu giao hàng đang mâu thuẫn nên cần kiểm tra riêng.
- **Giúp ích:** Tránh tự đổi trạng thái đơn chỉ vì có ngày giao.


In [ ]:
from pyspark.sql import functions as F

canceled_with_delivery = (
    orders_prepared
    .filter(
        (F.col("order_status") == "canceled") &
        F.col("order_delivered_customer_date").isNotNull()
    )
    .join(items_per_order, on="order_id", how="left")
    .join(payments_per_order, on="order_id", how="left")
    .withColumn(
        "so_ngay_tu_dat_den_giao_thuc_te",
        F.round(
            (
                F.unix_timestamp("order_delivered_customer_date")
                - F.unix_timestamp("order_purchase_timestamp")
            ) / 86400,
            2
        )
    )
    .withColumn(
        "giao_truoc_ngay_du_kien",
        F.when(
            F.to_date("order_delivered_customer_date")
            < F.to_date("order_estimated_delivery_date"),
            1
        ).otherwise(0)
    )
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "so_ngay_tu_dat_den_giao_thuc_te",
        "giao_truoc_ngay_du_kien",
        "so_dong_san_pham",
        "so_dong_thanh_toan",
        F.round("tong_gia_tri_thanh_toan", 2).alias("tong_gia_tri_thanh_toan"),
        "co_ban_giao_truoc_dat",
        "co_ban_giao_truoc_duyet",
        "co_giao_truoc_ban_giao"
    )
    .orderBy("order_purchase_timestamp")
)

print(f"SỐ ĐƠN CANCELED CÓ NGÀY GIAO: {canceled_with_delivery.count()}")
canceled_with_delivery.show(
    canceled_with_delivery.count(),
    truncate=False
)

SỐ ĐƠN CANCELED CÓ NGÀY GIAO: 6
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------------------+-----------------------+----------------+------------------+-----------------------+---------------------+-----------------------+----------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|so_ngay_tu_dat_den_giao_thuc_te|giao_truoc_ngay_du_kien|so_dong_san_pham|so_dong_thanh_toan|tong_gia_tri_thanh_toan|co_ban_giao_truoc_dat|co_ban_giao_truoc_duyet|co_giao_truoc_ban_giao|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+--------------

##Nhận xét:
- Cả 6 đơn canceled đều có sản phẩm, bản ghi thanh toán và chuỗi thời gian đặt → duyệt → bàn giao → giao khách đúng thứ tự
- 5 đơn được đặt trong tháng 10/2016, 1 đơn trong tháng 2/2018. Vì vậy, chúng là trường hợp mâu thuẫn giữa order_status và ngày giao

### Cell code 25: Gắn cờ đơn hủy có ngày giao

- **Mục đích:** Thêm cờ cho canceled có ngày giao và cập nhật tệp orders đã xử lý.
- **Vì sao:** Giữ nguyên trạng thái gốc nhưng vẫn ghi nhận trường hợp mâu thuẫn.
- **Giúp ích:** Có thể lọc hoặc phân tích độ nhạy của các đơn này về sau.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import shutil
from pyspark.sql import functions as F

output_path = (
    Path(customer_path).parent
    / "processed"
    / "olist_orders_prepared.csv"
)

orders_updated = orders_prepared.withColumn(
    "canceled_with_delivery_flag",
    F.when(
        (F.col("order_status") == "canceled") &
        F.col("order_delivered_customer_date").isNotNull(),
        1
    ).otherwise(0)
)

check = orders_updated.agg(
    F.count("*").alias("total_orders"),
    F.sum("canceled_with_delivery_flag").alias("flagged_orders")
).first()

if check["flagged_orders"] != 6:
    raise ValueError(
        f"Dự kiến 6 đơn được gắn cờ, thực tế: {check['flagged_orders']}"
    )

with TemporaryDirectory(prefix="olist_orders_flag_") as temporary:
    spark_dir = Path(temporary) / "spark_csv"

    (
        orders_updated
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(str(spark_dir))
    )

    part_file = next(spark_dir.glob("part-*.csv"))
    staged_file = output_path.with_name(output_path.name + ".tmp")
    shutil.copy2(part_file, staged_file)

    saved_check = (
        spark.read
        .option("header", True)
        .csv(str(staged_file))
        .agg(
            F.count("*").alias("total_orders"),
            F.sum(
                F.col("canceled_with_delivery_flag").cast("int")
            ).alias("flagged_orders")
        )
        .first()
    )

    if (
        saved_check["total_orders"] != check["total_orders"]
        or saved_check["flagged_orders"] != 6
    ):
        staged_file.unlink(missing_ok=True)
        raise ValueError("File xuất ra không khớp số dòng hoặc số đơn được gắn cờ")

    os.replace(staged_file, output_path)

orders_prepared = orders_updated

print(f"Đã cập nhật: {output_path}")
print(f"Tổng số đơn: {check['total_orders']}")
print(f"Đơn có canceled_with_delivery_flag = 1: {check['flagged_orders']}")

Đã cập nhật: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_orders_prepared.csv
Tổng số đơn: 99441
Đơn có canceled_with_delivery_flag = 1: 6


### Kiểm tra chất lượng bảng `order_items`

### Cell code 26: Kiểm tra chất lượng order_items

- **Mục đích:** Rà khóa, liên kết, giá, phí vận chuyển, ngày hạn bàn giao và số dòng trên đơn.
- **Vì sao:** Một đơn có nhiều sản phẩm là hợp lệ, nhưng trùng cặp khóa hoặc giá âm là vấn đề.
- **Giúp ích:** Đảm bảo bảng sản phẩm theo đơn đủ tin cậy để tạo feature.


In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

data_dir = Path(customer_path).parent

order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir / "olist_order_items_dataset.csv"))
)

products_check = (
    spark.read
    .option("header", True)
    .csv(str(data_dir / "olist_products_dataset.csv"))
    .select("product_id")
    .distinct()
)

orders_check = orders_prepared.select(
    "order_id",
    F.to_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp")
)

total_items = order_items.count()

missing_columns = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]

missing_values = order_items.agg(*[
    F.sum(
        F.when(
            F.col(column).isNull()
            | (F.trim(F.col(column).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(column)
    for column in missing_columns
]).first().asDict()

duplicate_rows = total_items - order_items.dropDuplicates().count()

duplicate_item_keys = (
    total_items
    - order_items.dropDuplicates(["order_id", "order_item_id"]).count()
)

orders_with_multiple_items = (
    order_items
    .filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

orders_not_found = (
    order_items
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
    .join(orders_check.select("order_id"), "order_id", "left_anti")
    .count()
)

products_not_found = (
    order_items
    .filter(F.col("product_id").isNotNull())
    .select("product_id")
    .distinct()
    .join(products_check, "product_id", "left_anti")
    .count()
)

items_audit = (
    order_items
    .withColumn("price_num", F.col("price").cast("double"))
    .withColumn("freight_num", F.col("freight_value").cast("double"))
    .withColumn("shipping_limit_ts", F.to_timestamp("shipping_limit_date"))
    .join(orders_check, "order_id", "left")
    .withColumn(
        "shipping_limit_before_purchase",
        F.col("shipping_limit_ts") < F.col("order_purchase_timestamp")
    )
)

numeric_time_summary = items_audit.agg(
    F.sum(F.when(F.col("price_num") < 0, 1).otherwise(0)).alias("price_am"),
    F.sum(F.when(F.col("price_num") == 0, 1).otherwise(0)).alias("price_bang_0"),
    F.sum(F.when(F.col("freight_num") < 0, 1).otherwise(0)).alias("freight_am"),
    F.sum(F.when(F.col("freight_num") == 0, 1).otherwise(0)).alias("freight_bang_0"),
    F.sum(
        F.when(
            F.col("price").isNotNull() & F.col("price_num").isNull(), 1
        ).otherwise(0)
    ).alias("price_khong_doc_duoc"),
    F.sum(
        F.when(
            F.col("freight_value").isNotNull() & F.col("freight_num").isNull(), 1
        ).otherwise(0)
    ).alias("freight_khong_doc_duoc"),
    F.sum(
        F.when(
            F.col("shipping_limit_date").isNotNull()
            & F.col("shipping_limit_ts").isNull(),
            1
        ).otherwise(0)
    ).alias("ngay_khong_doc_duoc"),
    F.sum(
        F.when(F.col("shipping_limit_before_purchase"), 1).otherwise(0)
    ).alias("han_ban_giao_truoc_ngay_dat")
).first().asDict()

checks = [
    ("Tổng số dòng sản phẩm", total_items),
    ("Dòng trùng hoàn toàn", duplicate_rows),
    ("Dòng dư do trùng cặp order_id–order_item_id", duplicate_item_keys),
    ("Số đơn có nhiều dòng sản phẩm", orders_with_multiple_items),
    ("Số order_id không tìm thấy trong orders", orders_not_found),
    ("Số product_id không tìm thấy trong products", products_not_found)
]

checks += [
    (f"Giá trị thiếu: {column}", missing_values[column])
    for column in missing_columns
]

checks += list(numeric_time_summary.items())

print("TỔNG QUAN CHẤT LƯỢNG ORDER_ITEMS")
spark.createDataFrame(
    [(name, int(value or 0)) for name, value in checks],
    ["kiem_tra", "so_luong"]
).show(50, truncate=False)

print("PHÂN BỐ GIÁ VÀ PHÍ VẬN CHUYỂN")
items_audit.select("price_num", "freight_num").summary(
    "count", "min", "50%", "99%", "max"
).show(truncate=False)

print("SỐ DÒNG SẢN PHẨM TRÊN MỖI ĐƠN")
order_items.groupBy("order_id").count().groupBy("count").agg(
    F.count("*").alias("so_don")
).orderBy("count").show(30, truncate=False)

print("MẪU DÒNG CÓ HẠN BÀN GIAO TRƯỚC NGÀY ĐẶT")
items_audit.filter(
    F.col("shipping_limit_before_purchase")
).select(
    "order_id",
    "order_item_id",
    "order_purchase_timestamp",
    "shipping_limit_date",
    "price",
    "freight_value"
).orderBy("order_id", "order_item_id").show(20, truncate=False)

TỔNG QUAN CHẤT LƯỢNG ORDER_ITEMS
+-------------------------------------------+--------+
|kiem_tra                                   |so_luong|
+-------------------------------------------+--------+
|Tổng số dòng sản phẩm                      |112650  |
|Dòng trùng hoàn toàn                       |0       |
|Dòng dư do trùng cặp order_id–order_item_id|0       |
|Số đơn có nhiều dòng sản phẩm              |9803    |
|Số order_id không tìm thấy trong orders    |0       |
|Số product_id không tìm thấy trong products|0       |
|Giá trị thiếu: order_id                    |0       |
|Giá trị thiếu: order_item_id               |0       |
|Giá trị thiếu: product_id                  |0       |
|Giá trị thiếu: seller_id                   |0       |
|Giá trị thiếu: shipping_limit_date         |0       |
|Giá trị thiếu: price                       |0       |
|Giá trị thiếu: freight_value               |0       |
|price_am                                   |0       |
|price_bang_0                   

Cell code: Trực quan hóa kiểm tra chất lượng và phân bố giá trị bảng order_items
Mục đích: Sử dụng hvplot biểu diễn các biểu đồ cột thống kê tổng quan vấn đề chất lượng (dòng trùng, thiếu dữ liệu, sai sót số học/thời gian), phân bố số dòng sản phẩm trên mỗi đơn hàng và phân vị các mức giá (price, freight_value).

Vì sao: Đánh giá độ tin cậy của bảng order_items, phát hiện các bất thường về giá/phí vận chuyển cũng như mức độ đa dạng chi tiết mặt hàng trong mỗi đơn.

Giúp ích: Cung cấp cơ sở thực tế để kiểm soát lỗi dữ liệu và phục vụ gom nhóm (aggregate) chính xác thông tin sản phẩm về cấp đơn hàng (order_id) ở các ô tiền xử lý tiếp theo.

In [ ]:
total_items_26a = order_items.count()

missing_columns_26a = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]

missing_values_26a = order_items.agg(*[
    F.sum(
        F.when(
            F.col(column).isNull()
            | (F.trim(F.col(column).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(column)
    for column in missing_columns_26a
]).first().asDict()

duplicate_rows_26a = (
    total_items_26a
    - order_items.dropDuplicates().count()
)

duplicate_item_keys_26a = (
    total_items_26a
    - order_items.dropDuplicates(
        ["order_id", "order_item_id"]
    ).count()
)

orders_with_multiple_items_26a = (
    order_items
    .filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

orders_not_found_26a = (
    order_items
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
    .join(
        orders_check.select("order_id"),
        "order_id",
        "left_anti"
    )
    .count()
)

products_not_found_26a = (
    order_items
    .filter(F.col("product_id").isNotNull())
    .select("product_id")
    .distinct()
    .join(
        products_check,
        "product_id",
        "left_anti"
    )
    .count()
)

numeric_time_summary_26a = items_audit.agg(
    F.sum(
        F.when(F.col("price_num") < 0, 1).otherwise(0)
    ).alias("Giá âm"),

    F.sum(
        F.when(F.col("price_num") == 0, 1).otherwise(0)
    ).alias("Giá bằng 0"),

    F.sum(
        F.when(F.col("freight_num") < 0, 1).otherwise(0)
    ).alias("Phí vận chuyển âm"),

    F.sum(
        F.when(F.col("freight_num") == 0, 1).otherwise(0)
    ).alias("Phí vận chuyển bằng 0"),

    F.sum(
        F.when(
            F.col("shipping_limit_before_purchase"),
            1
        ).otherwise(0)
    ).alias("Hạn bàn giao trước ngày đặt")
).first().asDict()

order_items_quality_rows = [
    ("Tổng số dòng sản phẩm", total_items_26a),
    ("Dòng trùng hoàn toàn", duplicate_rows_26a),
    (
        "Trùng order_id-order_item_id",
        duplicate_item_keys_26a
    ),
    (
        "Đơn có nhiều dòng sản phẩm",
        orders_with_multiple_items_26a
    ),
    (
        "Order ID không có trong orders",
        orders_not_found_26a
    ),
    (
        "Product ID không có trong products",
        products_not_found_26a
    )
]

order_items_quality_rows += [
    (
        f"Thiếu {column}",
        int(missing_values_26a[column] or 0)
    )
    for column in missing_columns_26a
]

order_items_quality_rows += [
    (name, int(value or 0))
    for name, value in numeric_time_summary_26a.items()
]

order_items_quality_pd = pd.DataFrame(
    order_items_quality_rows,
    columns=["Kiểm tra", "Số lượng"]
)

order_items_issue_pd = order_items_quality_pd[
    order_items_quality_pd["Số lượng"] > 0
].copy()

display(
    order_items_issue_pd.hvplot.bar(
        x="Kiểm tra",
        y="Số lượng",
        width=1200,
        height=470,
        rot=55,
        title="Tổng quan chất lượng order_items",
        ylabel="Số lượng"
    ).opts(
        hooks=[black_chart_text]
    )
)

item_count_pd = (
    order_items
    .groupBy("order_id")
    .count()
    .groupBy("count")
    .agg(
        F.count("*").alias("so_don")
    )
    .orderBy("count")
    .toPandas()
    .rename(columns={
        "count": "Số dòng sản phẩm",
        "so_don": "Số đơn"
    })
)

display(
    item_count_pd.hvplot.bar(
        x="Số dòng sản phẩm",
        y="Số đơn",
        width=900,
        height=420,
        title="Phân bố số dòng sản phẩm trên mỗi đơn",
        xlabel="Số dòng sản phẩm",
        ylabel="Số đơn"
    ).opts(
        hooks=[black_chart_text]
    )
)

price_summary_pd = (
    items_audit
    .select(
        "price_num",
        "freight_num"
    )
    .summary(
        "min",
        "50%",
        "99%",
        "max"
    )
    .toPandas()
)

price_summary_pd["price_num"] = pd.to_numeric(
    price_summary_pd["price_num"],
    errors="coerce"
)

price_summary_pd["freight_num"] = pd.to_numeric(
    price_summary_pd["freight_num"],
    errors="coerce"
)

display(
    price_summary_pd.hvplot.bar(
        x="summary",
        y=[
            "price_num",
            "freight_num"
        ],
        width=850,
        height=420,
        title="Phân vị giá sản phẩm và phí vận chuyển",
        xlabel="Thống kê",
        ylabel="Giá trị"
    ).opts(
        hooks=[black_chart_text]
    )
)

:Bars   [Kiểm tra]   (Số lượng)

:Bars   [Số dòng sản phẩm]   (Số đơn)

:Bars   [summary,Variable]   (value)

##Nhận xét:
- 112.650 dòng sản phẩm, không thiếu dữ liệu, không trùng khóa, các mã đơn và sản phẩm đều nối được, cũng không có hạn bàn giao trước ngày đặt.


### Cell code 27: Xem phí vận chuyển bằng 0 và giá cao

- **Mục đích:** Kiểm tra nhóm freight bằng 0 cùng các giá và phí ở phân vị cao, đối chiếu đơn/thanh toán.
- **Vì sao:** Giá trị bằng 0 hoặc lớn chưa tự chứng minh là lỗi.
- **Giúp ích:** Quyết định giữ hay đánh cờ dựa trên dữ liệu liên quan.


In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

data_dir = Path(customer_path).parent

payments_audit = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir / "olist_order_payments_dataset.csv"))
    .groupBy("order_id")
    .agg(
        F.count("*").alias("so_dong_thanh_toan"),
        F.round(
            F.sum(F.col("payment_value").cast("double")), 2
        ).alias("tong_thanh_toan")
    )
)

items_per_order_audit = (
    order_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("so_san_pham_trong_don"),
        F.round(F.sum(F.col("price").cast("double")), 2).alias("tong_gia_san_pham"),
        F.round(
            F.sum(F.col("freight_value").cast("double")), 2
        ).alias("tong_phi_van_chuyen")
    )
)

item_details = (
    order_items
    .withColumn("price_num", F.col("price").cast("double"))
    .withColumn("freight_num", F.col("freight_value").cast("double"))
    .join(
        orders_prepared.select("order_id", "order_status"),
        "order_id",
        "left"
    )
    .join(items_per_order_audit, "order_id", "left")
    .join(payments_audit, "order_id", "left")
)

p99_price = item_details.approxQuantile("price_num", [0.99], 0.001)[0]
p99_freight = item_details.approxQuantile("freight_num", [0.99], 0.001)[0]

zero_freight = item_details.filter(F.col("freight_num") == 0)

high_values = (
    item_details
    .filter(
        (F.col("price_num") > F.lit(p99_price))
        | (F.col("freight_num") > F.lit(p99_freight))
    )
    .withColumn(
        "loai_gia_tri_cao",
        F.when(
            (F.col("price_num") > F.lit(p99_price))
            & (F.col("freight_num") > F.lit(p99_freight)),
            "Giá và phí vận chuyển đều cao"
        )
        .when(
            F.col("price_num") > F.lit(p99_price),
            "Giá sản phẩm cao"
        )
        .otherwise("Phí vận chuyển cao")
    )
)

print("NGƯỠNG KIỂM TRA")
print("Phân vị 99% của price:", round(p99_price, 2))
print("Phân vị 99% của freight_value:", round(p99_freight, 2))

print("DÒNG CÓ PHÍ VẬN CHUYỂN BẰNG 0 THEO TRẠNG THÁI")
zero_freight.groupBy("order_status").agg(
    F.count("*").alias("so_dong_san_pham"),
    F.countDistinct("order_id").alias("so_don"),
    F.sum(
        F.when(F.col("tong_thanh_toan").isNull(), 1).otherwise(0)
    ).alias("so_dong_thieu_ban_ghi_thanh_toan"),
    F.round(F.min("price_num"), 2).alias("gia_thap_nhat"),
    F.round(F.max("price_num"), 2).alias("gia_cao_nhat")
).orderBy(F.desc("so_dong_san_pham")).show(truncate=False)

print("SỐ DÒNG CÓ GIÁ HOẶC PHÍ VẬN CHUYỂN CAO")
high_values.groupBy("loai_gia_tri_cao", "order_status").agg(
    F.count("*").alias("so_dong_san_pham"),
    F.countDistinct("order_id").alias("so_don"),
    F.round(F.max("price_num"), 2).alias("gia_cao_nhat"),
    F.round(F.max("freight_num"), 2).alias("phi_cao_nhat")
).orderBy("loai_gia_tri_cao", F.desc("so_dong_san_pham")).show(truncate=False)

columns_to_show = [
    "order_id",
    "order_item_id",
    "order_status",
    "product_id",
    "price",
    "freight_value",
    "so_san_pham_trong_don",
    "tong_gia_san_pham",
    "tong_phi_van_chuyen",
    "so_dong_thanh_toan",
    "tong_thanh_toan"
]

print("MẪU DÒNG PHÍ VẬN CHUYỂN BẰNG 0")
zero_freight.select(*columns_to_show).orderBy(
    F.desc("price_num")
).show(zero_freight.count(), truncate=False)

print("MẪU DÒNG CÓ GIÁ SẢN PHẨM CAO NHẤT")
high_values.filter(
    F.col("price_num") > F.lit(p99_price)
).select(*columns_to_show).orderBy(
    F.desc("price_num")
).show(high_values.count(), truncate=False)

print("MẪU DÒNG CÓ PHÍ VẬN CHUYỂN CAO NHẤT")
high_values.filter(
    F.col("freight_num") > F.lit(p99_freight)
).select(*columns_to_show).orderBy(
    F.desc("freight_num")
).show(high_values.count(), truncate=False)

NGƯỠNG KIỂM TRA
Phân vị 99% của price: 858.9
Phân vị 99% của freight_value: 82.5
DÒNG CÓ PHÍ VẬN CHUYỂN BẰNG 0 THEO TRẠNG THÁI
+------------+----------------+------+--------------------------------+-------------+------------+
|order_status|so_dong_san_pham|so_don|so_dong_thieu_ban_ghi_thanh_toan|gia_thap_nhat|gia_cao_nhat|
+------------+----------------+------+--------------------------------+-------------+------------+
|delivered   |381             |337   |0                               |53.9         |712.9       |
|shipped     |2               |2     |0                               |69.9         |99.9        |
+------------+----------------+------+--------------------------------+-------------+------------+

SỐ DÒNG CÓ GIÁ HOẶC PHÍ VẬN CHUYỂN CAO
+-----------------------------+------------+----------------+------+------------+------------+
|loai_gia_tri_cao             |order_status|so_dong_san_pham|so_don|gia_cao_nhat|phi_cao_nhat|
+-----------------------------+------------+-----

Cell code: Trực quan hóa phí vận chuyển bằng 0 và sản phẩm có giá trị cao theo trạng thái đơn
Mục đích: Sử dụng hvplot biểu diễn hai biểu đồ cột: một biểu đồ thống kê các đơn có phí vận chuyển bằng 0 (freight = 0) và một biểu đồ cột chồng (stacked bar chart) phân bố các dòng sản phẩm có giá trị bán/phí vận chuyển bất thường (ngoại lệ cao) theo từng order_status.

Vì sao: Đánh giá xem hiện tượng miễn phí vận chuyển hoặc các giá trị ngoại lệ (outliers) về giá/cước phí có tập trung vào các trạng thái đơn hàng cụ thể nào hay không (ví dụ: đơn hàng bị hủy hoặc hoàn tất).

Giúp ích: Cung cấp cơ sở thực tế để thiết lập quy tắc xử lý ngoại lệ tài chính và kiểm soát chất lượng dữ liệu giá trị đơn hàng trước khi chuyển sang bước tổng hợp đặc trưng.

In [ ]:
zero_freight_pd = (
    zero_freight
    .groupBy("order_status")
    .agg(
        F.count("*").alias("so_dong_san_pham"),
        F.countDistinct("order_id").alias("so_don")
    )
    .orderBy(F.desc("so_dong_san_pham"))
    .toPandas()
)

display(
    zero_freight_pd.hvplot.bar(
        x="order_status",
        y=["so_dong_san_pham", "so_don"],
        width=850,
        height=410,
        title="Phí vận chuyển bằng 0 theo trạng thái đơn",
        xlabel="Trạng thái",
        ylabel="Số lượng"
    ).opts(hooks=[black_chart_text])
)

high_value_pd = (
    high_values
    .groupBy("loai_gia_tri_cao", "order_status")
    .agg(
        F.count("*").alias("so_dong_san_pham")
    )
    .orderBy("loai_gia_tri_cao", F.desc("so_dong_san_pham"))
    .toPandas()
)

display(
    high_value_pd.hvplot.bar(
        x="order_status",
        y="so_dong_san_pham",
        by="loai_gia_tri_cao",
        stacked=True,
        width=1000,
        height=430,
        rot=30,
        title="Giá sản phẩm và phí vận chuyển cao theo trạng thái",
        xlabel="Trạng thái",
        ylabel="Số dòng sản phẩm"
    ).opts(hooks=[black_chart_text])
)

:Bars   [order_status,Variable]   (value)

:Bars   [order_status,loai_gia_tri_cao]   (so_dong_san_pham)

##Nhận xét:
- 383 dòng phí vận chuyển bằng 0 thuộc 339 đơn: 337 đơn delivered, 2 đơn shipped; không dòng nào thiếu bản ghi thanh toán. Nhiều mẫu có tổng thanh toán bằng giá sản phẩm, phù hợp với phí vận chuyển bằng 0.

### Cell code 28: Kiểm tra chất lượng payments

- **Mục đích:** Rà khóa, bản ghi thiếu/trùng, installments, giá trị và hình thức thanh toán.
- **Vì sao:** Một đơn có thể có nhiều dòng thanh toán; cần tránh coi đó là đơn lặp.
- **Giúp ích:** Biết khoản nào có thể dùng khi tính chi tiêu theo đơn.


In [ ]:
from pyspark.sql import functions as F

payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(data_dir / "olist_order_payments_dataset.csv"))
)

payments_checked = (
    payments
    .withColumn("payment_sequential_num", F.expr("try_cast(payment_sequential as int)"))
    .withColumn("payment_installments_num", F.expr("try_cast(payment_installments as int)"))
    .withColumn("payment_value_num", F.expr("try_cast(payment_value as decimal(18,2))"))
)

order_ids = orders_prepared.select("order_id").distinct()

duplicate_rows = (
    payments.groupBy(*payments.columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_keys = (
    payments_checked
    .filter(
        (F.col("order_id").isNotNull()) &
        (F.trim(F.col("order_id")) != "") &
        (F.col("payment_sequential_num").isNotNull())
    )
    .groupBy("order_id", "payment_sequential_num")
    .count()
    .filter(F.col("count") > 1)
)

payment_orders = (
    payments_checked
    .filter(
        (F.col("order_id").isNotNull()) &
        (F.trim(F.col("order_id")) != "")
    )
    .select("order_id")
    .distinct()
)

metrics = payments_checked.agg(
    F.count("*").alias("tong_so_dong"),
    F.countDistinct("order_id").alias("so_don_co_thanh_toan"),
    F.sum(F.when(F.col("order_id").isNull() | (F.trim("order_id") == ""), 1).otherwise(0)).alias("thieu_order_id"),
    F.sum(F.when(F.col("payment_sequential").isNull() | (F.trim("payment_sequential") == ""), 1).otherwise(0)).alias("thieu_payment_sequential"),
    F.sum(F.when(F.col("payment_installments").isNull() | (F.trim("payment_installments") == ""), 1).otherwise(0)).alias("thieu_payment_installments"),
    F.sum(F.when(F.col("payment_type").isNull() | (F.trim("payment_type") == ""), 1).otherwise(0)).alias("thieu_payment_type"),
    F.sum(F.when(F.col("payment_value").isNull() | (F.trim("payment_value") == ""), 1).otherwise(0)).alias("thieu_payment_value"),
    F.sum(F.when(
        F.col("payment_sequential").isNotNull() &
        (F.trim("payment_sequential") != "") &
        F.col("payment_sequential_num").isNull(), 1
    ).otherwise(0)).alias("sequential_khong_doc_duoc"),
    F.sum(F.when(
        F.col("payment_installments").isNotNull() &
        (F.trim("payment_installments") != "") &
        F.col("payment_installments_num").isNull(), 1
    ).otherwise(0)).alias("installments_khong_doc_duoc"),
    F.sum(F.when(
        F.col("payment_value").isNotNull() &
        (F.trim("payment_value") != "") &
        F.col("payment_value_num").isNull(), 1
    ).otherwise(0)).alias("payment_value_khong_doc_duoc"),
    F.sum(F.when(F.col("payment_sequential_num") <= 0, 1).otherwise(0)).alias("sequential_khong_duong"),
    F.sum(F.when(F.col("payment_installments_num") <= 0, 1).otherwise(0)).alias("installments_khong_duong"),
    F.sum(F.when(F.col("payment_value_num") < 0, 1).otherwise(0)).alias("payment_value_am"),
    F.sum(F.when(F.col("payment_value_num") == 0, 1).otherwise(0)).alias("payment_value_bang_0")
).first().asDict()

metrics.update({
    "dong_trung_hoan_toan": duplicate_rows.count(),
    "dong_du_do_trung_hoan_toan": duplicate_rows.agg(
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0))
    ).first()[0],
    "cap_order_id_sequential_bi_trung": duplicate_keys.count(),
    "dong_du_do_trung_cap_order_id_sequential": duplicate_keys.agg(
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0))
    ).first()[0],
    "dong_payment_khong_noi_duoc_orders": (
        payments_checked
        .filter(F.col("order_id").isNotNull() & (F.trim("order_id") != ""))
        .join(order_ids, "order_id", "left_anti")
        .count()
    ),
    "don_trong_orders_khong_co_payment": (
        order_ids.join(payment_orders, "order_id", "left_anti").count()
    ),
    "don_co_nhieu_dong_thanh_toan": (
        payments_checked
        .filter(F.col("order_id").isNotNull() & (F.trim("order_id") != ""))
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
})

check_rows = [(name, int(value or 0)) for name, value in metrics.items()]
check_df = spark.createDataFrame(check_rows, ["kiem_tra", "so_luong"])

print("TỔNG QUAN CHẤT LƯỢNG PAYMENTS")
check_df.show(len(check_rows), truncate=False)

print("PHÂN BỐ THEO HÌNH THỨC THANH TOÁN")
payments_checked.groupBy("payment_type").agg(
    F.count("*").alias("so_dong"),
    F.countDistinct("order_id").alias("so_don"),
    F.sum("payment_value_num").alias("tong_tien")
).orderBy(F.desc("so_dong")).show(truncate=False)

print("SỐ DÒNG THANH TOÁN TRÊN MỖI ĐƠN")
payments_checked.filter(
    F.col("order_id").isNotNull() & (F.trim("order_id") != "")
).groupBy("order_id").count().groupBy("count").agg(
    F.count("*").alias("so_don")
).orderBy("count").show(truncate=False)

print("PHÂN BỐ GIÁ TRỊ THANH TOÁN")
payments_checked.select("payment_value_num").summary(
    "count", "min", "50%", "99%", "max"
).show(truncate=False)

print("MẪU CẶP ORDER_ID–PAYMENT_SEQUENTIAL BỊ TRÙNG")
duplicate_keys.orderBy(F.desc("count")).show(20, truncate=False)

TỔNG QUAN CHẤT LƯỢNG PAYMENTS
+----------------------------------------+--------+
|kiem_tra                                |so_luong|
+----------------------------------------+--------+
|tong_so_dong                            |103886  |
|so_don_co_thanh_toan                    |99440   |
|thieu_order_id                          |0       |
|thieu_payment_sequential                |0       |
|thieu_payment_installments              |0       |
|thieu_payment_type                      |0       |
|thieu_payment_value                     |0       |
|sequential_khong_doc_duoc               |0       |
|installments_khong_doc_duoc             |0       |
|payment_value_khong_doc_duoc            |0       |
|sequential_khong_duong                  |0       |
|installments_khong_duong                |2       |
|payment_value_am                        |0       |
|payment_value_bang_0                    |9       |
|dong_trung_hoan_toan                    |0       |
|dong_du_do_trung_hoan_toan       

Cell code: Trực quan hóa kiểm tra chất lượng, hình thức thanh toán và phân bố dòng payment

Mục đích: Sử dụng hvplot biểu diễn 3 biểu đồ cột: các chỉ số kiểm tra chất lượng dữ liệu thanh toán khác 0, phân bố các phương thức thanh toán (payment_type), và tần suất số dòng thanh toán phát sinh trên mỗi đơn hàng.

Vì sao: Phát hiện nhanh các vấn đề bất thường trong bảng thanh toán, đồng thời đánh giá mức độ phổ biến của từng phương thức thanh toán và hành vi thanh toán nhiều lần/nhiều đợt của khách hàng.

Giúp ích: Cung cấp cơ sở thực tế để thiết lập các cờ chất lượng thanh toán (payment quality flags) và định hình logic gom nhóm dữ liệu payment chuẩn xác trước khi tính toán đặc trưng.

In [ ]:
payment_quality_pd = check_df.toPandas()

payment_quality_nonzero_pd = payment_quality_pd[
    payment_quality_pd["so_luong"] > 0
].copy()

display(
    payment_quality_nonzero_pd.hvplot.bar(
        x="kiem_tra",
        y="so_luong",
        width=1200,
        height=460,
        rot=55,
        title="Các chỉ số chất lượng payment có giá trị khác 0",
        xlabel="Kiểm tra",
        ylabel="Số lượng"
    ).opts(hooks=[black_chart_text])
)

payment_type_pd = (
    payments_checked
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("so_dong"),
        F.countDistinct("order_id").alias("so_don")
    )
    .orderBy(F.desc("so_dong"))
    .toPandas()
)

display(
    payment_type_pd.hvplot.bar(
        x="payment_type",
        y=["so_dong", "so_don"],
        width=900,
        height=420,
        rot=25,
        title="Phân bố hình thức thanh toán",
        xlabel="Hình thức thanh toán",
        ylabel="Số lượng"
    ).opts(hooks=[black_chart_text])
)

payment_count_pd = (
    payments_checked
    .filter(
        F.col("order_id").isNotNull()
        & (F.trim("order_id") != "")
    )
    .groupBy("order_id")
    .count()
    .groupBy("count")
    .agg(F.count("*").alias("so_don"))
    .orderBy("count")
    .toPandas()
    .rename(columns={
        "count": "Số dòng payment",
        "so_don": "Số đơn"
    })
)

display(
    payment_count_pd.hvplot.bar(
        x="Số dòng payment",
        y="Số đơn",
        width=800,
        height=400,
        title="Số dòng thanh toán trên mỗi đơn"
    ).opts(hooks=[black_chart_text])
)

:Bars   [kiem_tra]   (so_luong)

:Bars   [payment_type,Variable]   (value)

:Bars   [Số dòng payment]   (Số đơn)

### Cell code 29: Xem ngoại lệ thanh toán

- **Mục đích:** Kiểm tra đơn thiếu thanh toán, số kỳ bằng 0, tiền bằng 0 và khoản tiền lớn.
- **Vì sao:** Các trường hợp này có thể hợp lệ hoặc do lỗi nhập liệu.
- **Giúp ích:** Tập hợp ngoại lệ trước khi đối chiếu toàn bộ đơn.


In [ ]:
from pyspark.sql import functions as F

items_by_order = (
    order_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.sum(F.col("price").cast("decimal(18,2)")).alias("total_price"),
        F.sum(F.col("freight_value").cast("decimal(18,2)")).alias("total_freight")
    )
)

payments_by_order = (
    payments_checked
    .groupBy("order_id")
    .agg(
        F.count("*").alias("payment_count"),
        F.sum("payment_value_num").alias("total_payment")
    )
)

order_amount_audit = (
    orders_prepared
    .select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    )
    .join(items_by_order, "order_id", "left")
    .join(payments_by_order, "order_id", "left")
    .withColumn(
        "total_items_and_freight",
        F.col("total_price") + F.col("total_freight")
    )
    .withColumn(
        "payment_difference",
        F.col("total_payment") - F.col("total_items_and_freight")
    )
)

payment_row_details = (
    payments_checked
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value_num",
        "payment_installments_num"
    )
    .join(
        order_amount_audit.select(
            "order_id",
            "order_status",
            "item_count",
            "payment_count",
            "total_price",
            "total_freight",
            "total_items_and_freight",
            "total_payment",
            "payment_difference"
        ),
        "order_id",
        "left"
    )
)

print("1. ĐƠN KHÔNG CÓ BẢN GHI THANH TOÁN")
order_amount_audit.filter(
    F.col("payment_count").isNull()
).show(20, truncate=False)

print("2. CÁC DÒNG PAYMENT_INSTALLMENTS = 0")
payment_row_details.filter(
    F.col("payment_installments_num") == 0
).orderBy("order_id", "payment_sequential").show(30, truncate=False)

print("3. CÁC DÒNG PAYMENT_VALUE = 0")
payment_row_details.filter(
    F.col("payment_value_num") == 0
).orderBy("order_id", "payment_sequential").show(30, truncate=False)

p99_payment = payments_checked.agg(
    F.percentile_approx(
        "payment_value_num", 0.99, 10000
    ).alias("p99_payment")
).first()["p99_payment"]

large_payments = payment_row_details.filter(
    F.col("payment_value_num") >= F.lit(p99_payment)
)

print(f"4. KHOẢN THANH TOÁN LỚN: PAYMENT_VALUE TỪ {p99_payment} TRỞ LÊN")

large_payments.agg(
    F.count("*").alias("so_dong_thanh_toan"),
    F.countDistinct("order_id").alias("so_don"),
    F.min("payment_value_num").alias("gia_tri_nho_nhat"),
    F.max("payment_value_num").alias("gia_tri_lon_nhat")
).show(truncate=False)

print("PHÂN BỐ KHOẢN THANH TOÁN LỚN THEO TRẠNG THÁI VÀ HÌNH THỨC")
large_payments.groupBy(
    "order_status", "payment_type"
).agg(
    F.count("*").alias("so_dong"),
    F.countDistinct("order_id").alias("so_don")
).orderBy(
    F.desc("so_dong"), "order_status", "payment_type"
).show(30, truncate=False)

print("25 KHOẢN THANH TOÁN LỚN NHẤT")
large_payments.orderBy(
    F.desc("payment_value_num")
).show(25, truncate=False)

1. ĐƠN KHÔNG CÓ BẢN GHI THANH TOÁN
+--------------------------------+------------+------------------------+-----------------------------+----------+-----------+-------------+-------------+-------------+-----------------------+------------------+
|order_id                        |order_status|order_purchase_timestamp|order_delivered_customer_date|item_count|total_price|total_freight|payment_count|total_payment|total_items_and_freight|payment_difference|
+--------------------------------+------------+------------------------+-----------------------------+----------+-----------+-------------+-------------+-------------+-----------------------+------------------+
|bfbd0f9bdef84302105ad712db648a6c|delivered   |2016-09-15 12:16:38     |2016-11-09 07:47:38          |3         |134.97     |8.49         |NULL         |NULL         |143.46                 |NULL              |
+--------------------------------+------------+------------------------+-----------------------------+----------+--------

### Cell code 30: Đối chiếu tiền của toàn bộ đơn

- **Mục đích:** So tổng payment với tổng giá sản phẩm cộng freight theo order_id.
- **Vì sao:** Không thể suy kết luận từ một số đơn mẫu; cần kiểm tra toàn bộ.
- **Giúp ích:** Phân loại đơn khớp, lệch, thiếu sản phẩm hoặc thiếu thanh toán.


In [ ]:
from pyspark.sql import functions as F

payment_reconciliation = (
    order_amount_audit
    .withColumn(
        "reconciliation_status",
        F.when(
            F.col("item_count").isNull() & F.col("payment_count").isNull(),
            "THIEU_SAN_PHAM_VA_THANH_TOAN"
        )
        .when(F.col("item_count").isNull(), "THIEU_SAN_PHAM")
        .when(F.col("payment_count").isNull(), "THIEU_THANH_TOAN")
        .when(F.col("payment_difference") == 0, "KHOP")
        .otherwise("LECH")
    )
    .withColumn(
        "absolute_difference",
        F.abs(F.col("payment_difference"))
    )
)

total_orders = payment_reconciliation.count()

print(f"TỔNG SỐ ĐƠN ĐƯỢC ĐỐI CHIẾU: {total_orders}")

print("KẾT QUẢ ĐỐI CHIẾU TRÊN TOÀN BỘ ĐƠN")
payment_reconciliation.groupBy("reconciliation_status").agg(
    F.count("*").alias("so_don")
).withColumn(
    "ty_le_pct",
    F.round(F.col("so_don") * 100 / F.lit(total_orders), 2)
).orderBy("reconciliation_status").show(truncate=False)

print("KẾT QUẢ THEO TRẠNG THÁI ĐƠN")
payment_reconciliation.groupBy(
    "order_status", "reconciliation_status"
).agg(
    F.count("*").alias("so_don")
).orderBy(
    "order_status", "reconciliation_status"
).show(100, truncate=False)

print("MỨC CHÊNH LỆCH CỦA CÁC ĐƠN KHÔNG KHỚP")
payment_reconciliation.filter(
    F.col("reconciliation_status") == "LECH"
).withColumn(
    "difference_range",
    F.when(F.col("absolute_difference") <= 1, "01. Đến 1")
    .when(F.col("absolute_difference") <= 10, "02. Trên 1 đến 10")
    .when(F.col("absolute_difference") <= 100, "03. Trên 10 đến 100")
    .otherwise("04. Trên 100")
).groupBy("difference_range").agg(
    F.count("*").alias("so_don"),
    F.min("absolute_difference").alias("chenh_lech_nho_nhat"),
    F.max("absolute_difference").alias("chenh_lech_lon_nhat")
).orderBy("difference_range").show(truncate=False)

print("25 ĐƠN CÓ CHÊNH LỆCH LỚN NHẤT")
payment_reconciliation.filter(
    F.col("reconciliation_status") == "LECH"
).select(
    "order_id",
    "order_status",
    "item_count",
    "payment_count",
    "total_price",
    "total_freight",
    "total_items_and_freight",
    "total_payment",
    "payment_difference",
    "absolute_difference"
).orderBy(
    F.desc("absolute_difference")
).show(25, truncate=False)

TỔNG SỐ ĐƠN ĐƯỢC ĐỐI CHIẾU: 99441
KẾT QUẢ ĐỐI CHIẾU TRÊN TOÀN BỘ ĐƠN
+---------------------+------+---------+
|reconciliation_status|so_don|ty_le_pct|
+---------------------+------+---------+
|KHOP                 |98089 |98.64    |
|LECH                 |576   |0.58     |
|THIEU_SAN_PHAM       |775   |0.78     |
|THIEU_THANH_TOAN     |1     |0.0      |
+---------------------+------+---------+

KẾT QUẢ THEO TRẠNG THÁI ĐƠN
+------------+---------------------+------+
|order_status|reconciliation_status|so_don|
+------------+---------------------+------+
|approved    |KHOP                 |2     |
|canceled    |KHOP                 |459   |
|canceled    |LECH                 |2     |
|canceled    |THIEU_SAN_PHAM       |164   |
|created     |THIEU_SAN_PHAM       |5     |
|delivered   |KHOP                 |95908 |
|delivered   |LECH                 |569   |
|delivered   |THIEU_THANH_TOAN     |1     |
|invoiced    |KHOP                 |311   |
|invoiced    |LECH                 |1     |
|i

Cell code: Trực quan hóa kết quả đối chiếu tiền thanh toán và mức độ chênh lệch

Mục đích: Sử dụng hvplot biểu diễn 3 biểu đồ cột: kết quả đối chiếu thanh toán tổng quan (reconciliation_status), chi tiết đối chiếu tiền theo từng order_status (dạng stacked bar), và khoảng phân bố giá trị chênh lệch cho các đơn hàng bị lệch tiền (LECH).

Vì sao: Đánh giá mức độ khớp tiền giữa tổng giá trị thanh toán và tổng giá trị đơn hàng, đồng thời làm rõ tỷ lệ sai lệch tập trung ở các đơn hàng đã hủy/giao hay do chênh lệch làm tròn nhỏ.

Giúp ích: Cung cấp bằng chứng thực tế về độ tin cậy của dữ liệu tài chính, giúp đưa ra quyết định xử lý hoặc loại bỏ các đơn sai lệch giá trị nghiêm trọng trước khi bước vào Feature Engineering.

In [ ]:
reconciliation_pd = (
    payment_reconciliation
    .groupBy("reconciliation_status")
    .agg(F.count("*").alias("so_don"))
    .withColumn(
        "ty_le_pct",
        F.round(F.col("so_don") * 100 / F.lit(total_orders), 2)
    )
    .orderBy(F.desc("so_don"))
    .toPandas()
)

display(
    reconciliation_pd.hvplot.bar(
        x="reconciliation_status",
        y="so_don",
        width=900,
        height=420,
        rot=20,
        title="Kết quả đối chiếu thanh toán toàn bộ đơn",
        xlabel="Kết quả đối chiếu",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

reconciliation_status_pd = (
    payment_reconciliation
    .groupBy("order_status", "reconciliation_status")
    .agg(F.count("*").alias("so_don"))
    .toPandas()
)

display(
    reconciliation_status_pd.hvplot.bar(
        x="order_status",
        y="so_don",
        by="reconciliation_status",
        stacked=True,
        width=1050,
        height=450,
        rot=30,
        title="Kết quả đối chiếu tiền theo trạng thái đơn",
        xlabel="Trạng thái đơn",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

difference_range_pd = (
    payment_reconciliation
    .filter(F.col("reconciliation_status") == "LECH")
    .withColumn(
        "difference_range",
        F.when(F.col("absolute_difference") <= 1, "01. Đến 1")
        .when(F.col("absolute_difference") <= 10, "02. Trên 1 đến 10")
        .when(F.col("absolute_difference") <= 100, "03. Trên 10 đến 100")
        .otherwise("04. Trên 100")
    )
    .groupBy("difference_range")
    .agg(F.count("*").alias("so_don"))
    .orderBy("difference_range")
    .toPandas()
)

display(
    difference_range_pd.hvplot.bar(
        x="difference_range",
        y="so_don",
        width=850,
        height=400,
        rot=20,
        title="Phân bố mức chênh lệch của các đơn không khớp",
        xlabel="Mức chênh lệch",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

:Bars   [reconciliation_status]   (so_don)

:Bars   [order_status,reconciliation_status]   (so_don)

:Bars   [difference_range]   (so_don)

##Nhận xét:
- 98.089/99.441 đơn (98,64%) có tổng thanh toán khớp đúng với giá sản phẩm cộng phí vận chuyển.
- Có 576 đơn lệch tiền (0,58%), trong đó 569 đơn đã giao. Mức lệch trải từ 0,01 đến 182,81; 327 đơn lệch không quá 1. 775 đơn thiếu dòng sản phẩm, chủ yếu thuộc nhóm unavailable (603) và canceled (164).

### Cell code 31: Phân tích các đơn lệch tiền

- **Mục đích:** Đếm mức lệch theo trạng thái, kiểu thanh toán và xem nhóm lệch trên 10.
- **Vì sao:** Cần biết lỗi tập trung ở đâu trước khi dùng tiền làm feature.
- **Giúp ích:** Định lượng phạm vi ảnh hưởng của các đơn không khớp.


In [ ]:
from pyspark.sql import functions as F

payment_methods_by_order = (
    payments_checked
    .groupBy("order_id")
    .agg(
        F.concat_ws(
            " + ",
            F.sort_array(F.collect_set("payment_type"))
        ).alias("payment_types")
    )
)

reconciliation_details = (
    payment_reconciliation
    .join(payment_methods_by_order, "order_id", "left")
)

mismatched_orders = reconciliation_details.filter(
    F.col("reconciliation_status") == "LECH"
)

large_mismatches = mismatched_orders.filter(
    F.col("absolute_difference") > 10
)

print("PHẠM VI ẢNH HƯỞNG CỦA CÁC ĐƠN LỆCH")
mismatched_orders.agg(
    F.count("*").alias("so_don_lech"),
    F.sum(
        F.when(F.col("absolute_difference") > 10, 1).otherwise(0)
    ).alias("so_don_lech_tren_10"),
    F.sum(
        F.when(F.col("payment_difference") > 0, 1).otherwise(0)
    ).alias("so_don_thanh_toan_cao_hon"),
    F.sum(
        F.when(F.col("payment_difference") < 0, 1).otherwise(0)
    ).alias("so_don_thanh_toan_thap_hon"),
    F.round(F.sum("absolute_difference"), 2).alias("tong_chenh_lech_tuyet_doi"),
    F.round(F.sum("payment_difference"), 2).alias("chenh_lech_rong"),
    F.max("absolute_difference").alias("chenh_lech_lon_nhat")
).show(truncate=False)

print("ĐƠN LỆCH THEO TRẠNG THÁI")
status_totals = reconciliation_details.groupBy("order_status").agg(
    F.count("*").alias("tong_so_don")
)

mismatched_orders.groupBy("order_status").agg(
    F.count("*").alias("so_don_lech"),
    F.sum(
        F.when(F.col("absolute_difference") > 10, 1).otherwise(0)
    ).alias("so_don_lech_tren_10"),
    F.round(F.sum("absolute_difference"), 2).alias("tong_chenh_lech_tuyet_doi")
).join(
    status_totals, "order_status"
).withColumn(
    "ty_le_don_lech_pct",
    F.round(F.col("so_don_lech") * 100 / F.col("tong_so_don"), 2)
).orderBy(
    F.desc("so_don_lech")
).show(truncate=False)

print("ĐƠN LỆCH THEO HÌNH THỨC THANH TOÁN")
mismatched_orders.groupBy("payment_types").agg(
    F.count("*").alias("so_don_lech"),
    F.sum(
        F.when(F.col("absolute_difference") > 10, 1).otherwise(0)
    ).alias("so_don_lech_tren_10"),
    F.round(F.sum("absolute_difference"), 2).alias("tong_chenh_lech_tuyet_doi")
).orderBy(
    F.desc("so_don_lech")
).show(30, truncate=False)

print("98 ĐƠN LỆCH TRÊN 10 THEO TRẠNG THÁI VÀ HÌNH THỨC THANH TOÁN")
large_mismatches.groupBy(
    "order_status", "payment_types"
).agg(
    F.count("*").alias("so_don"),
    F.round(F.sum("absolute_difference"), 2).alias("tong_chenh_lech_tuyet_doi"),
    F.max("absolute_difference").alias("chenh_lech_lon_nhat")
).orderBy(
    F.desc("so_don")
).show(50, truncate=False)

print("CHI TIẾT TOÀN BỘ ĐƠN LỆCH TRÊN 10")
large_mismatches.select(
    "order_id",
    "order_status",
    "payment_types",
    "item_count",
    "payment_count",
    "total_price",
    "total_freight",
    "total_items_and_freight",
    "total_payment",
    "payment_difference",
    "absolute_difference"
).orderBy(
    F.desc("absolute_difference")
).show(100, truncate=False)

PHẠM VI ẢNH HƯỞNG CỦA CÁC ĐƠN LỆCH
+-----------+-------------------+-------------------------+--------------------------+-------------------------+---------------+-------------------+
|so_don_lech|so_don_lech_tren_10|so_don_thanh_toan_cao_hon|so_don_thanh_toan_thap_hon|tong_chenh_lech_tuyet_doi|chenh_lech_rong|chenh_lech_lon_nhat|
+-----------+-------------------+-------------------------+--------------------------+-------------------------+---------------+-------------------+
|576        |98                 |367                      |209                       |3271.95                  |2870.39        |182.81             |
+-----------+-------------------+-------------------------+--------------------------+-------------------------+---------------+-------------------+

ĐƠN LỆCH THEO TRẠNG THÁI
+------------+-----------+-------------------+-------------------------+-----------+------------------+
|order_status|so_don_lech|so_don_lech_tren_10|tong_chenh_lech_tuyet_doi|tong_so_don|ty_le_

Cell code: Trực quan hóa tỷ lệ đơn lệch tiền theo trạng thái và hình thức thanh toán

Mục đích: Sử dụng hvplot biểu diễn 2 biểu đồ cột: tỷ lệ phần trăm đơn hàng bị sai lệch giá trị thanh toán (ty_le_don_lech_pct) theo từng order_status và thống kê chi tiết số đơn lệch tiền (bao gồm cả các đơn lệch đáng kể > 10) theo sự kết hợp các phương thức thanh toán (payment_types).

Vì sao: Xác định chính xác các nhóm trạng thái đơn hàng hoặc hình thức thanh toán nào có nguy cơ phát sinh sai lệch tiền cao nhất (ví dụ: các đơn hủy, đơn thanh toán kết hợp nhiều phương thức).

Giúp ích: Cung cấp căn cứ chi tiết để phân loại nguyên nhân sai lệch tài chính, hỗ trợ tinh chỉnh quy tắc lọc bớt dữ liệu rác trước khi tiến hành tạo đặc trưng (Feature Engineering).

In [ ]:
mismatch_status_pd = (
    mismatched_orders
    .groupBy("order_status")
    .agg(
        F.count("*").alias("so_don_lech"),
        F.sum(
            F.when(F.col("absolute_difference") > 10, 1).otherwise(0)
        ).alias("so_don_lech_tren_10")
    )
    .join(status_totals, "order_status")
    .withColumn(
        "ty_le_don_lech_pct",
        F.round(
            F.col("so_don_lech") * 100 / F.col("tong_so_don"),
            2
        )
    )
    .orderBy(F.desc("so_don_lech"))
    .toPandas()
)

display(
    mismatch_status_pd.hvplot.bar(
        x="order_status",
        y="ty_le_don_lech_pct",
        width=900,
        height=420,
        rot=25,
        title="Tỷ lệ đơn lệch tiền theo trạng thái",
        xlabel="Trạng thái",
        ylabel="Tỷ lệ đơn lệch (%)"
    ).opts(hooks=[black_chart_text])
)

mismatch_payment_pd = (
    mismatched_orders
    .groupBy("payment_types")
    .agg(
        F.count("*").alias("so_don_lech"),
        F.sum(
            F.when(F.col("absolute_difference") > 10, 1).otherwise(0)
        ).alias("so_don_lech_tren_10")
    )
    .orderBy(F.desc("so_don_lech"))
    .toPandas()
)

display(
    mismatch_payment_pd.hvplot.bar(
        x="payment_types",
        y=["so_don_lech", "so_don_lech_tren_10"],
        width=1000,
        height=430,
        rot=35,
        title="Đơn lệch tiền theo hình thức thanh toán",
        xlabel="Hình thức thanh toán",
        ylabel="Số đơn"
    ).opts(hooks=[black_chart_text])
)

:Bars   [order_status]   (ty_le_don_lech_pct)

:Bars   [payment_types,Variable]   (value)

##Nhận xét:
- 576/99.441 đơn (0,58%). Trong đó 98 đơn lệch trên 10; nhóm này gần như tập trung ở đơn đã giao và thanh toán bằng thẻ tín dụng (95 đơn).
- Ở 576 đơn lệch, có 367 đơn thanh toán cao hơn tiền hàng cộng phí vận chuyển và 209 đơn thấp hơn; tổng chênh lệch tuyệt đối là 3.271,95

### Cell code 32: Kiểm tra chất lượng products

- **Mục đích:** Rà product_id, danh mục, mô tả, cân nặng, kích thước và khóa nối với items.
- **Vì sao:** Thuộc tính thiếu hoặc bằng 0 có thể làm sai đặc trưng sản phẩm.
- **Giúp ích:** Xác định các trường cần cờ chất lượng khi tổng hợp theo đơn.


In [ ]:
from pyspark.sql import functions as F

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(data_dir / "olist_products_dataset.csv"))
)

numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_checked = products

for column in numeric_columns:
    products_checked = products_checked.withColumn(
        f"{column}_num",
        F.expr(f"try_cast({column} as double)")
    )

product_ids = (
    products
    .filter(F.col("product_id").isNotNull() & (F.trim("product_id") != ""))
    .select("product_id")
    .distinct()
)

item_product_ids = (
    order_items
    .filter(F.col("product_id").isNotNull() & (F.trim("product_id") != ""))
    .select("product_id")
    .distinct()
)

duplicate_rows = (
    products.groupBy(*products.columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_ids = (
    products
    .filter(F.col("product_id").isNotNull() & (F.trim("product_id") != ""))
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

checks = [
    F.count("*").alias("tong_so_san_pham"),
    F.countDistinct("product_id").alias("so_product_id_khac_nhau"),
    F.sum(F.when(
        F.col("product_id").isNull() | (F.trim("product_id") == ""), 1
    ).otherwise(0)).alias("thieu_product_id"),
    F.sum(F.when(
        F.col("product_category_name").isNull() |
        (F.trim("product_category_name") == ""), 1
    ).otherwise(0)).alias("thieu_product_category_name")
]

for column in numeric_columns:
    missing = F.col(column).isNull() | (F.trim(F.col(column)) == "")

    checks.extend([
        F.sum(F.when(missing, 1).otherwise(0)).alias(f"thieu_{column}"),
        F.sum(F.when(
            ~missing & F.col(f"{column}_num").isNull(), 1
        ).otherwise(0)).alias(f"khong_doc_duoc_{column}")
    ])

for column in [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]:
    checks.append(
        F.sum(F.when(
            F.col(f"{column}_num") <= 0, 1
        ).otherwise(0)).alias(f"{column}_khong_duong")
    )

checks.extend([
    F.sum(F.when(
        F.col("product_photos_qty_num") < 0, 1
    ).otherwise(0)).alias("product_photos_qty_am"),
    F.sum(F.when(
        F.col("product_photos_qty_num") == 0, 1
    ).otherwise(0)).alias("product_photos_qty_bang_0")
])

metrics = products_checked.agg(*checks).first().asDict()

metrics.update({
    "nhom_dong_trung_hoan_toan": duplicate_rows.count(),
    "dong_du_do_trung_hoan_toan": duplicate_rows.agg(
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0))
    ).first()[0],
    "product_id_bi_trung": duplicate_ids.count(),
    "dong_du_do_trung_product_id": duplicate_ids.agg(
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0))
    ).first()[0],
    "product_id_trong_order_items_khong_co_trong_products": (
        item_product_ids.join(product_ids, "product_id", "left_anti").count()
    ),
    "product_id_trong_products_chua_xuat_hien_o_order_items": (
        product_ids.join(item_product_ids, "product_id", "left_anti").count()
    )
})

result_rows = [(name, int(value or 0)) for name, value in metrics.items()]
result_df = spark.createDataFrame(result_rows, ["kiem_tra", "so_luong"])

print("TỔNG QUAN CHẤT LƯỢNG PRODUCTS")
result_df.show(len(result_rows), truncate=False)

print("15 DANH MỤC CÓ NHIỀU SẢN PHẨM NHẤT")
products.groupBy("product_category_name").agg(
    F.count("*").alias("so_san_pham")
).orderBy(F.desc("so_san_pham")).show(15, truncate=False)

print("PHÂN BỐ CÂN NẶNG VÀ KÍCH THƯỚC")
products_checked.select(
    "product_weight_g_num",
    "product_length_cm_num",
    "product_height_cm_num",
    "product_width_cm_num"
).summary("count", "min", "50%", "99%", "max").show(truncate=False)

print("MẪU PRODUCT_ID BỊ TRÙNG")
duplicate_ids.orderBy(F.desc("count")).show(20, truncate=False)

TỔNG QUAN CHẤT LƯỢNG PRODUCTS
+------------------------------------------------------+--------+
|kiem_tra                                              |so_luong|
+------------------------------------------------------+--------+
|tong_so_san_pham                                      |32951   |
|so_product_id_khac_nhau                               |32951   |
|thieu_product_id                                      |0       |
|thieu_product_category_name                           |610     |
|thieu_product_name_lenght                             |610     |
|khong_doc_duoc_product_name_lenght                    |0       |
|thieu_product_description_lenght                      |610     |
|khong_doc_duoc_product_description_lenght             |0       |
|thieu_product_photos_qty                              |610     |
|khong_doc_duoc_product_photos_qty                     |0       |
|thieu_product_weight_g                                |2       |
|khong_doc_duoc_product_weight_g              

Cell code: Trực quan hóa các vấn đề chất lượng và top danh mục sản phẩm của bảng products

Mục đích: Sử dụng hvplot để vẽ 2 biểu đồ cột: một biểu đồ thống kê các chỉ số kiểm tra chất lượng dữ liệu của bảng products có phát sinh lỗi (so_luong > 0), và một biểu đồ hiển thị top 15 danh mục sản phẩm sở hữu số lượng sản phẩm lớn nhất.

Vì sao: Phát hiện các khiếm khuyết trong thuộc tính sản phẩm (như thiếu tên danh mục, kích thước, trọng lượng) và nắm bắt cấu trúc phân bố hàng hóa theo ngành hàng.

Giúp ích: Cung cấp thông tin nền tảng để đánh giá độ sạch của bảng products và đưa ra chiến lược xử lý dữ liệu khuyết (imputation) hoặc gom nhóm danh mục trước khi tích hợp vào feature store.

In [ ]:
product_quality_pd = result_df.toPandas()

product_quality_nonzero_pd = product_quality_pd[
    product_quality_pd["so_luong"] > 0
].copy()

display(
    product_quality_nonzero_pd.hvplot.bar(
        x="kiem_tra",
        y="so_luong",
        width=1200,
        height=470,
        rot=55,
        title="Các vấn đề chất lượng của bảng products",
        xlabel="Kiểm tra",
        ylabel="Số lượng"
    ).opts(hooks=[black_chart_text])
)

top_categories_pd = (
    products
    .filter(F.col("product_category_name").isNotNull())
    .groupBy("product_category_name")
    .agg(F.count("*").alias("so_san_pham"))
    .orderBy(F.desc("so_san_pham"))
    .limit(15)
    .toPandas()
)

display(
    top_categories_pd.hvplot.bar(
        x="product_category_name",
        y="so_san_pham",
        width=1100,
        height=450,
        rot=45,
        title="15 danh mục có nhiều sản phẩm nhất",
        xlabel="Danh mục sản phẩm",
        ylabel="Số sản phẩm"
    ).opts(hooks=[black_chart_text])
)

:Bars   [kiem_tra]   (so_luong)

:Bars   [product_category_name]   (so_san_pham)

### Cell code 33: Lưu tiền theo đơn đã đối chiếu

- **Mục đích:** Tạo một dòng mỗi order_id với tổng tiền và cờ khớp, lệch hoặc thiếu.
- **Vì sao:** Giữ kết quả kiểm tra tiền ở cấp đơn giúp nối an toàn.
- **Giúp ích:** Có bảng chi tiêu chuẩn bị sẵn cho EDA và feature engineering.


In [ ]:
# ============================================================
# VALIDATE ORDER_PAYMENTS_PREPARED
#
# Mục tiêu:
# - Sử dụng trực tiếp DataFrame order_payments_prepared
#   đã được tạo ở cell trước.
# - KHÔNG đọc lại file CSV từ thư mục processed.
# - Kiểm tra dataset đã đạt grain:
#       1 dòng = 1 order
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 1. KIỂM TRA DATAFRAME CÓ TỒN TẠI KHÔNG
# ============================================================

if "order_payments_prepared" not in globals():
    raise RuntimeError(
        "Không tìm thấy DataFrame 'order_payments_prepared'.\n"
        "Hãy chạy các cell phía trên trước để tạo "
        "order_payments_prepared."
    )


print("=" * 70)
print("VALIDATE: order_payments_prepared")
print("=" * 70)


# ============================================================
# 2. KIỂM TRA SỐ DÒNG VÀ ORDER DUY NHẤT
# ============================================================

prepared_rows = order_payments_prepared.count()

prepared_unique_orders = (
    order_payments_prepared
    .select("order_id")
    .distinct()
    .count()
)

prepared_duplicate_orders = (
    prepared_rows - prepared_unique_orders
)


print("\n[1] KIỂM TRA GRAIN")
print("-" * 50)

print(
    f"Tổng số dòng         : {prepared_rows:,}"
)

print(
    f"Số order_id duy nhất : {prepared_unique_orders:,}"
)

print(
    f"Số order bị trùng    : {prepared_duplicate_orders:,}"
)


# ============================================================
# 3. VALIDATE GRAIN
# ============================================================

if prepared_rows != prepared_unique_orders:
    raise ValueError(
        "\norder_payments_prepared CHƯA ĐẠT grain "
        "'1 dòng = 1 order'.\n"
        f"Phát hiện {prepared_duplicate_orders:,} dòng dư "
        "do order_id bị lặp."
    )

print(
    "\n✓ ĐẠT: order_payments_prepared có grain "
    "1 dòng = 1 order"
)


# ============================================================
# 4. KIỂM TRA NULL ORDER_ID
# ============================================================

null_order_id = (
    order_payments_prepared
    .filter(
        F.col("order_id").isNull()
    )
    .count()
)


print("\n[2] KIỂM TRA ORDER_ID")
print("-" * 50)

print(
    f"order_id bị null      : {null_order_id:,}"
)


if null_order_id == 0:
    print(
        "✓ Không có order_id bị thiếu"
    )
else:
    print(
        "⚠ Có order_id bị thiếu, cần kiểm tra lại"
    )


# ============================================================
# 5. KIỂM TRA CÁC CỘT QUAN TRỌNG
# ============================================================

important_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "item_count",
    "payment_count",
    "total_price",
    "total_freight",
    "total_items_and_freight",
    "total_payment",
    "payment_difference",
    "absolute_difference",
    "reconciliation_status"
]


existing_columns = set(
    order_payments_prepared.columns
)

missing_columns = [
    col
    for col in important_columns
    if col not in existing_columns
]


print("\n[3] KIỂM TRA CÁC CỘT CẦN THIẾT")
print("-" * 50)


if len(missing_columns) == 0:

    print(
        "✓ Có đầy đủ các cột order/payment cần thiết"
    )

else:

    print(
        "⚠ Thiếu các cột:"
    )

    for col in missing_columns:
        print(
            f"  - {col}"
        )


# ============================================================
# 6. HIỂN THỊ SCHEMA
# ============================================================

print("\n[4] SCHEMA")
print("-" * 50)

order_payments_prepared.printSchema()


# ============================================================
# 7. HIỂN THỊ MỘT SỐ DÒNG MẪU
# ============================================================

print("\n[5] SAMPLE DATA")
print("-" * 50)


sample_columns = [
    col
    for col in [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "item_count",
        "payment_count",
        "total_price",
        "total_freight",
        "total_payment",
        "reconciliation_status"
    ]
    if col in existing_columns
]


order_payments_prepared.select(
    *sample_columns
).show(
    10,
    truncate=False
)


# ============================================================
# 8. KẾT LUẬN
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION COMPLETED")
print("=" * 70)

print(
    f"✓ Rows          : {prepared_rows:,}"
)

print(
    f"✓ Unique orders : {prepared_unique_orders:,}"
)

print(
    f"✓ Duplicate     : {prepared_duplicate_orders:,}"
)

print(
    "✓ Grain         : 1 row = 1 order"
)

print(
    "\norder_payments_prepared đã sẵn sàng "
    "cho bước controlled join tiếp theo."
)

VALIDATE: order_payments_prepared

[1] KIỂM TRA GRAIN
--------------------------------------------------
Tổng số dòng         : 99,441
Số order_id duy nhất : 99,441
Số order bị trùng    : 0

✓ ĐẠT: order_payments_prepared có grain 1 dòng = 1 order

[2] KIỂM TRA ORDER_ID
--------------------------------------------------
order_id bị null      : 0
✓ Không có order_id bị thiếu

[3] KIỂM TRA CÁC CỘT CẦN THIẾT
--------------------------------------------------
✓ Có đầy đủ các cột order/payment cần thiết

[4] SCHEMA
--------------------------------------------------
root
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- item_count: integer (nullable = true)
 |-- payment_count: integer (nullable = true)
 |-- total_price: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- total_items_and_freight: double (nullable = true)
 |-- total_payment: double (nullable = true)
 |-- pay



### Cell code 34: Lưu sản phẩm với cờ chất lượng

- **Mục đích:** Gắn cờ danh mục/kích thước/cân nặng thiếu và cân nặng bằng 0 rồi lưu CSV.
- **Vì sao:** Không nên tự điền các thông số sản phẩm khi chưa có căn cứ.
- **Giúp ích:** Có thể tổng hợp đặc trưng sản phẩm và tách trường hợp thiếu rõ ràng.


In [ ]:
# ============================================================
# VALIDATE PRODUCTS_PREPARED
#
# Mục tiêu:
# - Sử dụng trực tiếp DataFrame products_prepared
#   đã được tạo ở cell phía trên.
# - KHÔNG đọc lại olist_products_prepared.csv.
# - Kiểm tra grain ở cấp product.
# ============================================================

from pyspark.sql import functions as F


# ============================================================
# 1. KIỂM TRA DATAFRAME CÓ TỒN TẠI
# ============================================================

if "products_prepared" not in globals():
    raise RuntimeError(
        "Không tìm thấy DataFrame 'products_prepared'.\n"
        "Hãy chạy các cell phía trên trước để tạo products_prepared."
    )


print("=" * 70)
print("VALIDATE: products_prepared")
print("=" * 70)


# ============================================================
# 2. KIỂM TRA GRAIN
# ============================================================

product_rows = products_prepared.count()

unique_products = (
    products_prepared
    .select("product_id")
    .distinct()
    .count()
)

duplicate_products = product_rows - unique_products


print("\n[1] KIỂM TRA GRAIN")
print("-" * 50)

print(f"Tổng số dòng          : {product_rows:,}")
print(f"Số product_id duy nhất: {unique_products:,}")
print(f"Số product bị trùng   : {duplicate_products:,}")


if product_rows != unique_products:
    raise ValueError(
        "products_prepared chưa đạt grain "
        "'1 dòng = 1 product'. "
        f"Có {duplicate_products:,} dòng dư."
    )

print("\n✓ ĐẠT: 1 dòng = 1 product")


# ============================================================
# 3. KIỂM TRA NULL PRODUCT_ID
# ============================================================

null_product_id = (
    products_prepared
    .filter(
        F.col("product_id").isNull()
    )
    .count()
)


print("\n[2] KIỂM TRA PRODUCT_ID")
print("-" * 50)

print(f"product_id bị null: {null_product_id:,}")


if null_product_id == 0:
    print("✓ Không có product_id bị thiếu")
else:
    print("⚠ Có product_id bị thiếu")


# ============================================================
# 4. KIỂM TRA CÁC QUALITY FLAGS
# ============================================================

quality_flags = [
    "product_category_missing_flag",
    "product_dimensions_missing_flag",
    "product_weight_missing_flag",
    "product_weight_zero_flag"
]

existing_columns = set(products_prepared.columns)

existing_flags = [
    c for c in quality_flags
    if c in existing_columns
]


print("\n[3] PRODUCT QUALITY FLAGS")
print("-" * 50)


if existing_flags:

    for flag in existing_flags:

        flagged_count = (
            products_prepared
            .filter(
                F.col(flag).cast("int") == 1
            )
            .count()
        )

        percentage = (
            flagged_count / product_rows * 100
            if product_rows > 0
            else 0
        )

        print(
            f"{flag:<35} "
            f"{flagged_count:>8,} "
            f"({percentage:.2f}%)"
        )

else:

    print(
        "Không tìm thấy quality flag trong products_prepared."
    )


# ============================================================
# 5. KIỂM TRA CÁC CỘT PRODUCT CƠ BẢN
# ============================================================

important_columns = [
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_columns = [
    c for c in important_columns
    if c not in existing_columns
]


print("\n[4] KIỂM TRA CỘT")
print("-" * 50)


if len(missing_columns) == 0:

    print("✓ Có đầy đủ các cột product chính")

else:

    print("⚠ Không tìm thấy một số cột:")

    for c in missing_columns:
        print(f"  - {c}")


# ============================================================
# 6. SCHEMA
# ============================================================

print("\n[5] SCHEMA")
print("-" * 50)

products_prepared.printSchema()


# ============================================================
# 7. SAMPLE
# ============================================================

print("\n[6] SAMPLE DATA")
print("-" * 50)


sample_columns = [
    c
    for c in [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_category_missing_flag",
        "product_dimensions_missing_flag",
        "product_weight_missing_flag",
        "product_weight_zero_flag"
    ]
    if c in existing_columns
]


products_prepared.select(
    *sample_columns
).show(
    10,
    truncate=False
)


# ============================================================
# 8. KẾT LUẬN
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION COMPLETED")
print("=" * 70)

print(f"✓ Rows            : {product_rows:,}")
print(f"✓ Unique products : {unique_products:,}")
print(f"✓ Duplicate       : {duplicate_products:,}")
print("✓ Grain           : 1 row = 1 product")

print(
    "\nproducts_prepared đã sẵn sàng "
    "cho các bước phân tích tiếp theo."
)

VALIDATE: products_prepared

[1] KIỂM TRA GRAIN
--------------------------------------------------
Tổng số dòng          : 32,951
Số product_id duy nhất: 32,951
Số product bị trùng   : 0

✓ ĐẠT: 1 dòng = 1 product

[2] KIỂM TRA PRODUCT_ID
--------------------------------------------------
product_id bị null: 0
✓ Không có product_id bị thiếu

[3] PRODUCT QUALITY FLAGS
--------------------------------------------------
product_category_missing_flag            610 (1.85%)
product_dimensions_missing_flag            2 (0.01%)
product_weight_missing_flag                2 (0.01%)
product_weight_zero_flag                   4 (0.01%)

[4] KIỂM TRA CỘT
--------------------------------------------------
✓ Có đầy đủ các cột product chính

[5] SCHEMA
--------------------------------------------------
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullabl

Cell code: Trực quan hóa kết quả gắn cờ chất lượng của products_prepared

Mục đích: Đếm số lượng sản phẩm gặp các vấn đề về dữ liệu (thiếu danh mục, thiếu kích thước, thiếu hoặc bằng 0 về cân nặng) từ dataframe products_prepared và biểu diễn dưới dạng biểu đồ cột bằng hvplot.

Vì sao: Đánh giá mức độ hoàn thiện của bảng sản phẩm sau khi đã qua bước chuẩn hóa dữ liệu sơ bộ, giúp phát hiện cụ thể nhóm thuộc tính nào vẫn còn khuyết thiếu.

Giúp ích: Cung cấp số liệu cụ thể để đưa ra chiến lược xử lý phù hợp (như điền giá trị mặc định/trung vị hoặc gắn cờ cảnh báo) trước khi đưa các thuộc tính sản phẩm vào tính toán đặc trưng.

In [ ]:
product_flags_data = []

for flag in [
    "product_category_missing_flag",
    "product_dimensions_missing_flag",
    "product_weight_missing_flag",
    "product_weight_zero_flag"
]:
    if flag in products_prepared.columns:
        count_flag = (
            products_prepared
            .filter(F.col(flag).cast("int") == 1)
            .count()
        )

        product_flags_data.append(
            (flag, count_flag)
        )

product_flags_pd = pd.DataFrame(
    product_flags_data,
    columns=["Quality flag", "Số sản phẩm"]
)

product_flags_pd["Quality flag"] = product_flags_pd["Quality flag"].replace({
    "product_category_missing_flag": "Thiếu danh mục",
    "product_dimensions_missing_flag": "Thiếu kích thước",
    "product_weight_missing_flag": "Thiếu cân nặng",
    "product_weight_zero_flag": "Cân nặng bằng 0"
})

display(
    product_flags_pd.hvplot.bar(
        x="Quality flag",
        y="Số sản phẩm",
        width=850,
        height=410,
        rot=20,
        title="Quality flags của products_prepared"
    ).opts(hooks=[black_chart_text])
)

:Bars   [Quality flag]   (Số sản phẩm)

### Cell code 35: Tạo bảng giao dịch chuẩn phục vụ EDA

- **Mục đích:** Chuyển bước ghép dữ liệu nền ra khỏi notebook EDA. Ghép đơn `delivered` với khách hàng, tiền theo đơn và thông tin sản phẩm đã tổng hợp, đồng thời kiểm soát số dòng sau từng phép nối.
- **Grain:** 1 dòng = 1 `order_id`.
- **Đầu ra:** `processed/olist_eda_delivered_orders.csv`, dùng trực tiếp cho EDA.


In [ ]:
# ============================================================
# TẠO VÀ LƯU BẢNG NỀN EDA - GRAIN: 1 ROW = 1 DELIVERED ORDER
# ============================================================

from pyspark.sql import functions as F
from pathlib import Path

# 1. Lấy các đơn delivered
delivered_orders = orders_prepared.filter(
    F.col("order_status") == "delivered"
).select(
    "order_id", "customer_id", "order_status", "order_purchase_timestamp"
)

n_delivered = delivered_orders.count()

# 2. Nối thông tin khách hàng
customer_map = customers.select("customer_id", "customer_unique_id")

if customer_map.count() != customer_map.select("customer_id").distinct().count():
    raise ValueError("customer_id trong customers không duy nhất.")

orders_with_customer = delivered_orders.join(
    customer_map, on="customer_id", how="left"
)
n_after_customer = orders_with_customer.count()

# 3. Kiểm tra và nối payment/order information
payment_rows = order_payments_prepared.count()
payment_unique = order_payments_prepared.select("order_id").distinct().count()

if payment_rows != payment_unique:
    raise ValueError("order_payments_prepared chưa đạt grain 1 dòng = 1 order.")

payment_cols = [
    "order_id", "item_count", "payment_count", "total_price",
    "total_freight", "total_items_and_freight", "total_payment",
    "payment_difference", "absolute_difference", "reconciliation_status",
    "payment_matched_flag", "payment_mismatch_flag",
    "items_missing_flag", "payment_missing_flag"
]
payment_cols = [c for c in payment_cols if c in order_payments_prepared.columns]

payment_by_order = order_payments_prepared.select(*payment_cols)

orders_with_payment = orders_with_customer.join(
    payment_by_order, on="order_id", how="left"
)
n_after_payment = orders_with_payment.count()

# 4. Tổng hợp chất lượng sản phẩm về cấp order
product_cols = ["product_id", "product_category_missing_flag"]
product_cols = [c for c in product_cols if c in products_prepared.columns]

product_quality = order_items.select(
    "order_id", "product_id"
).join(
    products_prepared.select(*product_cols),
    on="product_id",
    how="left"
)

product_by_order = product_quality.groupBy("order_id").agg(
    F.count("*").alias("product_item_rows"),
    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("missing_product_rows"),
    F.sum(
        F.when(F.col("product_category_missing_flag").cast("int") == 1, 1).otherwise(0)
    ).alias("missing_category_item_count")
).withColumn(
    "has_missing_product_category",
    F.when(F.col("missing_category_item_count") > 0, 1).otherwise(0)
)

if product_by_order.count() != product_by_order.select("order_id").distinct().count():
    raise ValueError("product_by_order chưa đạt grain 1 dòng = 1 order.")

# 5. Nối product information vào bảng cuối
eda_orders = orders_with_payment.join(
    product_by_order, on="order_id", how="left"
)
n_after_product = eda_orders.count()

# 6. Kiểm tra số dòng sau từng lần nối
join_check = spark.createDataFrame([
    ("Đơn delivered ban đầu", n_delivered),
    ("Sau khi nối customers", n_after_customer),
    ("Sau khi nối tiền theo đơn", n_after_payment),
    ("Sau khi nối sản phẩm theo đơn", n_after_product)
], ["buoc_noi", "so_dong"])

print("KIỂM TRA SỐ DÒNG SAU TỪNG LẦN NỐI")
join_check.show(truncate=False)

if len({n_delivered, n_after_customer, n_after_payment, n_after_product}) != 1:
    raise ValueError("Số dòng thay đổi sau join, có khả năng xảy ra row multiplication.")

print("✓ Không xảy ra row multiplication.")

# 7. Tổng quan dữ liệu nền EDA
if "payment_mismatch_flag" in eda_orders.columns:
    mismatch_cond = F.col("payment_mismatch_flag").cast("int") == 1
elif "reconciliation_status" in eda_orders.columns:
    mismatch_cond = F.lower(F.col("reconciliation_status")).contains("mismatch")
else:
    mismatch_cond = F.lit(False)

if "reconciliation_status" in eda_orders.columns:
    missing_payment_cond = F.col("reconciliation_status").isNull()
else:
    missing_payment_cond = F.lit(False)

eda_summary = eda_orders.agg(
    F.count("*").alias("so_don_delivered"),
    F.countDistinct("order_id").alias("so_order_id_duy_nhat"),
    F.countDistinct("customer_unique_id").alias("so_khach"),
    F.sum(F.when(F.col("customer_unique_id").isNull(), 1).otherwise(0))
        .alias("don_thieu_customer_unique_id"),
    F.sum(F.when(F.col("order_purchase_timestamp").isNull(), 1).otherwise(0))
        .alias("don_thieu_ngay_mua"),
    F.sum(F.when(missing_payment_cond, 1).otherwise(0))
        .alias("don_thieu_doi_chieu_tien"),
    F.sum(F.when(F.col("product_item_rows").isNull(), 1).otherwise(0))
        .alias("don_thieu_dong_san_pham"),
    F.sum(F.when(mismatch_cond, 1).otherwise(0))
        .alias("don_lech_tien"),
    F.sum(F.when(F.col("has_missing_product_category") == 1, 1).otherwise(0))
        .alias("don_co_san_pham_thieu_danh_muc")
)

print("\nTỔNG QUAN DỮ LIỆU NỀN EDA")
eda_summary.show(truncate=False)

# 8. Kiểm tra grain cuối cùng
final_rows = eda_orders.count()
final_unique_orders = eda_orders.select("order_id").distinct().count()
duplicate_orders = final_rows - final_unique_orders

print("\nKIỂM TRA GRAIN CUỐI")
print(f"Số dòng              : {final_rows:,}")
print(f"Số order_id duy nhất : {final_unique_orders:,}")
print(f"Số order bị trùng    : {duplicate_orders:,}")

if final_rows != final_unique_orders:
    raise ValueError("Bảng EDA chưa đạt grain 1 dòng = 1 order.")

missing_customer = eda_orders.filter(
    F.col("customer_unique_id").isNull()
).count()

missing_time = eda_orders.filter(
    F.col("order_purchase_timestamp").isNull()
).count()

print(f"Thiếu customer_unique_id : {missing_customer:,}")
print(f"Thiếu purchase timestamp : {missing_time:,}")
print("✓ Đạt grain: 1 dòng = 1 delivered order")

# 9. Lưu bằng chuẩn Spark: output là một thư mục chứa các part CSV
processed_dir = Path(data_dir) / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir = processed_dir / "olist_eda_delivered_orders"

(
    eda_orders.write
    .mode("overwrite")
    .option("header", True)
    .csv(str(output_dir))
)

# 10. Đọc lại trực tiếp thư mục Spark vừa ghi để kiểm tra
saved_eda_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(output_dir))
)

saved_rows = saved_eda_orders.count()
saved_unique_orders = saved_eda_orders.select("order_id").distinct().count()

print("\nKIỂM TRA SAU KHI LƯU")
print(f"Số dòng đã lưu       : {saved_rows:,}")
print(f"Order ID duy nhất    : {saved_unique_orders:,}")

if saved_rows != final_rows:
    raise ValueError("Số dòng sau khi lưu không khớp dữ liệu ban đầu.")

if saved_unique_orders != final_unique_orders:
    raise ValueError("Số order_id sau khi lưu không khớp dữ liệu ban đầu.")

print("\n" + "=" * 60)
print("HOÀN TẤT TẠO BẢNG NỀN EDA")
print("=" * 60)
print(f"✓ Delivered orders : {final_rows:,}")
print(f"✓ Unique orders    : {final_unique_orders:,}")
print(f"✓ Customers        : {eda_orders.select('customer_unique_id').distinct().count():,}")
print("✓ Grain            : 1 row = 1 delivered order")
print("✓ Row multiplication: không phát hiện")
print("✓ Dataset đã lưu tại:")
print(output_dir)

KIỂM TRA SỐ DÒNG SAU TỪNG LẦN NỐI
+-----------------------------+-------+
|buoc_noi                     |so_dong|
+-----------------------------+-------+
|Đơn delivered ban đầu        |96478  |
|Sau khi nối customers        |96478  |
|Sau khi nối tiền theo đơn    |96478  |
|Sau khi nối sản phẩm theo đơn|96478  |
+-----------------------------+-------+

✓ Không xảy ra row multiplication.

TỔNG QUAN DỮ LIỆU NỀN EDA
+----------------+--------------------+--------+----------------------------+------------------+------------------------+-----------------------+-------------+------------------------------+
|so_don_delivered|so_order_id_duy_nhat|so_khach|don_thieu_customer_unique_id|don_thieu_ngay_mua|don_thieu_doi_chieu_tien|don_thieu_dong_san_pham|don_lech_tien|don_co_san_pham_thieu_danh_muc|
+----------------+--------------------+--------+----------------------------+------------------+------------------------+-----------------------+-------------+------------------------------+
|96478    

Cell code: Trực quan hóa tiến trình Controlled Join và tổng quan chất lượng bảng giao dịch nền EDA

Mục đích: Sử dụng hvplot biểu diễn biểu đồ đường theo dõi sự thay đổi số lượng dòng qua từng bước kết nối kiểm soát (controlled join) và biểu đồ cột thống kê các chỉ số cảnh báo chất lượng (thiếu ID, thiếu ngày mua, lệch tiền, thiếu danh mục sản phẩm) trên tập dữ liệu tổng hợp phục vụ EDA.

Vì sao: Kiểm soát nguy cơ bùng nổ dữ liệu (data explosion) hoặc thất thoát bản ghi trong quá trình join nhiều bảng, đồng thời rà soát lần cuối các cờ chất lượng trên bảng dữ liệu hợp nhất.

Giúp ích: Đảm bảo độ tin cậy và tính toàn vẹn của bảng dữ liệu giao dịch nền trước khi chuyển sang các bước phân tích EDA chuyên sâu và trích xuất đặc trưng (Feature Engineering).

In [ ]:
join_check_pd = join_check.toPandas()

display(
    join_check_pd.hvplot.line(
        x="buoc_noi",
        y="so_dong",
        marker="circle",
        width=1000,
        height=410,
        rot=25,
        title="Số dòng sau từng bước controlled join",
        xlabel="Bước nối",
        ylabel="Số dòng"
    ).opts(hooks=[black_chart_text])
)

eda_summary_pd = (
    eda_summary
    .toPandas()
    .T
    .reset_index()
)

eda_summary_pd.columns = ["Chỉ số", "Số lượng"]

eda_quality_pd = eda_summary_pd[
    eda_summary_pd["Chỉ số"].isin([
        "don_thieu_customer_unique_id",
        "don_thieu_ngay_mua",
        "don_thieu_doi_chieu_tien",
        "don_thieu_dong_san_pham",
        "don_lech_tien",
        "don_co_san_pham_thieu_danh_muc"
    ])
].copy()

eda_quality_pd["Chỉ số"] = eda_quality_pd["Chỉ số"].replace({
    "don_thieu_customer_unique_id": "Thiếu customer_unique_id",
    "don_thieu_ngay_mua": "Thiếu ngày mua",
    "don_thieu_doi_chieu_tien": "Thiếu đối chiếu tiền",
    "don_thieu_dong_san_pham": "Thiếu dòng sản phẩm",
    "don_lech_tien": "Đơn lệch tiền",
    "don_co_san_pham_thieu_danh_muc": "Đơn có sản phẩm thiếu danh mục"
})

eda_quality_pd["Số lượng"] = pd.to_numeric(
    eda_quality_pd["Số lượng"]
)

display(
    eda_quality_pd.hvplot.bar(
        x="Chỉ số",
        y="Số lượng",
        width=1050,
        height=430,
        rot=35,
        title="Chất lượng bảng giao dịch nền phục vụ EDA"
    ).opts(hooks=[black_chart_text])
)

:Curve   [buoc_noi]   (so_dong)

:Bars   [Chỉ số]   (Số lượng)